In [1]:
# !pip install -q pandas numpy scikit-learn plotly openai litellm

import pandas as pd
import numpy as np
import datetime
import plotly.express as px
import plotly.graph_objects as go

import zipfile
import urllib.request
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, confusion_matrix
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge

from scipy import stats
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN

from sklearn.ensemble import IsolationForest
import json

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


# Set seed for reproducible synthetic generation
np.random.seed(42)
print("Environment successfully setup and dependencies loaded!")

Environment successfully setup and dependencies loaded!


In [2]:
# Create data directory
os.makedirs("cms_data", exist_ok=True)

print("1. Downloading CMS DE-SynPUF Sample 1 Files...")

# Official CMS Public Download URLs for Sample 1
bene_url = "https://www.cms.gov/Research-Statistics-Data-and-Systems/Downloadable-Public-Use-Files/SynPUFs/Downloads/DE1_0_2008_Beneficiary_Summary_File_Sample_1.zip"
carrier_url = "https://downloads.cms.gov/files/DE1_0_2008_to_2010_Carrier_Claims_Sample_1A.zip"

# Download Beneficiary File
urllib.request.urlretrieve(bene_url, "cms_data/bene_2008.zip")
with zipfile.ZipFile("cms_data/bene_2008.zip", 'r') as zip_ref:
    zip_ref.extractall("cms_data")

# Download Carrier Claims File
urllib.request.urlretrieve(carrier_url, "cms_data/carrier_claims.zip")
with zipfile.ZipFile("cms_data/carrier_claims.zip", 'r') as zip_ref:
    zip_ref.extractall("cms_data")

print("Files downloaded and unzipped successfully!")

1. Downloading CMS DE-SynPUF Sample 1 Files...
Files downloaded and unzipped successfully!


In [3]:
# Find extracted CSV file paths
bene_csv = [f for f in os.listdir("cms_data") if "Beneficiary" in f and f.endswith(".csv")][0]
carrier_csv = [f for f in os.listdir("cms_data") if "Carrier" in f and f.endswith(".csv")][0]

print("2. Loading CSVs into DataFrames...")
# Load a subset of 10,000 rows for high-speed prototyping in Colab
bene_df = pd.read_csv(os.path.join("cms_data", bene_csv), nrows=5000)
carrier_df = pd.read_csv(os.path.join("cms_data", carrier_csv), nrows=10000)

2. Loading CSVs into DataFrames...


/tmp/ipykernel_2157/3005902342.py:8: DtypeWarning: Columns (10,11,50,138,139,140,141) have mixed types. Specify dtype option on import or set low_memory=False.
  carrier_df = pd.read_csv(os.path.join("cms_data", carrier_csv), nrows=10000)


In [4]:
bene_df.head()

,DESYNPUF_ID,BENE_BIRTH_DT,BENE_DEATH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,BENE_ESRD_IND,SP_STATE_CODE,BENE_COUNTY_CD,BENE_HI_CVRAGE_TOT_MONS,BENE_SMI_CVRAGE_TOT_MONS,...,SP_STRKETIA,MEDREIMB_IP,BENRES_IP,PPPYMT_IP,MEDREIMB_OP,BENRES_OP,PPPYMT_OP,MEDREIMB_CAR,BENRES_CAR,PPPYMT_CAR
0,00013D2EFD8E45D1,19230501,NaN,1,1,0,26,950,12,12,...,2,0.0,0.0,0.0,50.0,10.0,0.0,0.0,0.0,0.0
1,00016F745862898F,19430101,NaN,1,1,0,39,230,12,12,...,2,0.0,0.0,0.0,0.0,0.0,0.0,700.0,240.0,0.0
2,0001FDD721E223DC,19360901,NaN,2,1,0,39,280,12,12,...,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,00021CA6FF03E670,19410601,NaN,1,5,0,6,290,0,0,...,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,00024B3D2352D2D0,19360801,NaN,1,1,0,52,590,12,12,...,2,0.0,0.0,0.0,30.0,40.0,0.0,220.0,80.0,0.0


In [5]:
carrier_df.head()

,DESYNPUF_ID,CLM_ID,CLM_FROM_DT,CLM_THRU_DT,ICD9_DGNS_CD_1,ICD9_DGNS_CD_2,ICD9_DGNS_CD_3,ICD9_DGNS_CD_4,ICD9_DGNS_CD_5,ICD9_DGNS_CD_6,...,LINE_ICD9_DGNS_CD_4,LINE_ICD9_DGNS_CD_5,LINE_ICD9_DGNS_CD_6,LINE_ICD9_DGNS_CD_7,LINE_ICD9_DGNS_CD_8,LINE_ICD9_DGNS_CD_9,LINE_ICD9_DGNS_CD_10,LINE_ICD9_DGNS_CD_11,LINE_ICD9_DGNS_CD_12,LINE_ICD9_DGNS_CD_13
0,00013D2EFD8E45D1,887733386680966,20090725,20090725,7245,7244,6272,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00013D2EFD8E45D1,887213386947664,20091014,20091014,3598,27541,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00013D2EFD8E45D1,887243388666441,20100401,20100401,29606,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00013D2EFD8E45D1,887893388307089,20100817,20100817,8410,8472,8409,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00013D2EFD8E45D1,887463387476539,20101105,20101105,29521,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Stage 1: Clean Merge (11 Chronic Conditions + Gender) & Multi-Provider Anomaly Injection

In [6]:
# 1. Define Complete Feature Sets
bene_cols = [
    'DESYNPUF_ID',
    'BENE_BIRTH_DT',
    'BENE_SEX_IDENT_CD',  # Gender: 1 = Male, 2 = Female
    'SP_STATE_CODE',       # Beneficiary State Location
    # ALL 11 CMS DE-SynPUF Chronic Condition Indicators
    'SP_ALZHDMTA',        # Alzheimer's / Dementia
    'SP_CHF',             # Congestive Heart Failure
    'SP_CHRNKIDN',        # Chronic Kidney Disease
    'SP_CNCR',            # Cancer
    'SP_COPD',            # Chronic Obstructive Pulmonary Disease
    'SP_DEPRESSN',        # Depression
    'SP_DIABETES',        # Diabetes
    'SP_ISCHMCHT',        # Ischemic Heart Disease
    'SP_OSTEOPRS',        # Osteoporosis
    'SP_RA_OA',           # Rheumatoid Arthritis / Osteoarthritis
    'SP_STRKETIA'         # Stroke / Transient Ischemic Attack
]

carrier_cols = [
    'DESYNPUF_ID',
    'CLM_ID',
    'CLM_FROM_DT',
    'PRF_PHYSN_NPI_1',
    'HCPCS_CD_1',
    'LINE_NCH_PMT_AMT_1'
]


In [7]:
# 2. Execute Inner Join
merged_cms_df = pd.merge(
    carrier_df[carrier_cols],
    bene_df[bene_cols],
    on="DESYNPUF_ID",
    how="inner"
)

# Standardize Date Column & Fill NA values
merged_cms_df['ORIGINAL_PMT_AMT'] = merged_cms_df['LINE_NCH_PMT_AMT_1'].copy()
merged_cms_df["CLM_FROM_DT"] = pd.to_datetime(merged_cms_df["CLM_FROM_DT"].astype(str), format="%Y%m%d")
merged_cms_df['LINE_NCH_PMT_AMT_1'] = pd.to_numeric(merged_cms_df['LINE_NCH_PMT_AMT_1'], errors='coerce').fillna(0.0)

# Stage 2: Kmeans Clustering and Anomoly injection

In [8]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')

# =========================================================================
# STEP 0: DYNAMIC COLUMN DETECTION & SAFETY CHECKS
# =========================================================================

def detect_dataframe_columns(df: pd.DataFrame) -> dict:
    """
    Dynamically identifies available CMS column equivalents.
    Returns a dictionary of mapped column names and flags missing essential fields.
    """
    candidates = {
        'bene_id': ['DESY_SORT_KEY', 'BENE_ID', 'BENE_IDENTIFIER', 'BENEFICIARY_ID'],
        'claim_id': ['CLM_ID', 'DESY_SORT_KEY', 'CLAIM_ID', 'CLM_NUM'],
        'provider_npi': ['PRF_PHYSN_NPI_1', 'RNDRNG_NPI', 'PRF_NPI', 'PRVDR_NUM', 'PROVIDER_NPI', 'NPI'],
        'hcpcs_cd': ['HCPCS_CD_1', 'ORIGINAL_HCPCS_CD', 'HCPCS_CD', 'CPT_CD', 'HCPCS'],
        'service_date': ['CLM_FROM_DT', 'SERVICE_DT', 'CLAIM_DATE', 'CLM_FROM_DATE'],
        'payment_amt': ['LINE_NCH_PMT_AMT_1', 'CLM_PMT_AMT', 'PAYMENT_AMOUNT', 'LINE_PAYMENT_AMT', 'ORIGINAL_PMT_AMT'],
        'specialty': ['PROVIDER_SPECIALTY', 'PRF_PHYSN_SPEC', 'SPECIALTY_CD'],
        'pos': ['PLACE_OF_SERVICE', 'POS_CD', 'LINE_PLACE_OF_SRVC_CD'],
        'line_num': ['LINE_NUM', 'CLM_LINE_NUM', 'CLAIM_LINE_NUMBER', 'LINE_INDEX']
    }

    mapped = {}
    for key, choices in candidates.items():
        found = next((col for col in choices if col in df.columns), None)
        mapped[key] = found

    # Essential column validation
    essential = ['claim_id', 'provider_npi', 'hcpcs_cd', 'service_date', 'payment_amt']
    missing_essential = [k for k in essential if mapped[k] is None]

    if missing_essential:
        raise KeyError(
            f"CRITICAL ERROR: Missing essential columns for mapping: {missing_essential}. "
            f"Available columns in dataframe: {list(df.columns)}"
        )

    return mapped


def largest_remainder_allocation(target_total: int, eligible_counts: dict) -> dict:
    """
    Allocates target scenario counts proportionally across clusters using the
    largest-remainder method to ensure exact integer summation and proportional representation.
    """
    total_eligible = sum(eligible_counts.values())
    if total_eligible == 0 or target_total == 0:
        return {k: 0 for k in eligible_counts}

    # Calculate exact proportional shares
    exact_shares = {k: (v / total_eligible) * target_total for k, v in eligible_counts.items()}
    allocated = {k: int(np.floor(v)) for k, v in exact_shares.items()}
    remainders = {k: v - allocated[k] for k, v in exact_shares.items()}

    # Distribute remainders
    remaining_needed = target_total - sum(allocated.values())
    sorted_keys = sorted(remainders.keys(), key=lambda k: remainders[k], reverse=True)

    for i in range(remaining_needed):
        allocated[sorted_keys[i % len(sorted_keys)]] += 1

    # Cap allocations by available eligible candidates in each cluster
    excess = 0
    for k in allocated:
        if allocated[k] > eligible_counts[k]:
            excess += allocated[k] - eligible_counts[k]
            allocated[k] = eligible_counts[k]

    # Reallocate excess if possible to other clusters with remaining capacity
    if excess > 0:
        for k in sorted_keys:
            can_take = eligible_counts[k] - allocated[k]
            add = min(excess, can_take)
            allocated[k] += add
            excess -= add
            if excess == 0:
                break

    return allocated


def run_pipeline(merged_cms_df: pd.DataFrame) -> pd.DataFrame:
    print("=" * 80)
    print("STARTING SYNTHETIC PAYMENT INTEGRITY DATA PROCESSING & SCENARIO INJECTION")
    print("=" * 80)

    # Detect mapped columns
    cols = detect_dataframe_columns(merged_cms_df)

    # Reset index safely
    df = merged_cms_df.copy().reset_index(drop=True)

    # =========================================================================
    # TASK 1: PRESERVE IMMUTABLE ORIGINAL & WORKING COLUMNS
    # =========================================================================
    print("\n[Task 1] Preserving immutable original baseline columns & working fields...")

    df['ORIGINAL_PMT_AMT'] = pd.to_numeric(df[cols['payment_amt']], errors='coerce').fillna(0.0)
    df['ORIGINAL_HCPCS_CD'] = df[cols['hcpcs_cd']].astype(str).str.strip()
    df['ORIGINAL_FROM_DT'] = pd.to_datetime(df[cols['service_date']], errors='coerce')
    df['ORIGINAL_PROVIDER_NPI'] = df[cols['provider_npi']].astype(str).str.strip()

    # Working fields for scenario modifications
    df['WORKING_PMT_AMT'] = df['ORIGINAL_PMT_AMT'].copy()
    df['WORKING_HCPCS_CD'] = df['ORIGINAL_HCPCS_CD'].copy()
    df['WORKING_FROM_DT'] = df['ORIGINAL_FROM_DT'].copy()
    df['WORKING_PROVIDER_NPI'] = df['ORIGINAL_PROVIDER_NPI'].copy()

    # =========================================================================
    # TASK 2: LINE-LEVEL IDENTIFIERS, DATA QUALITY & SPARSE PROVIDERS
    # =========================================================================
    print("[Task 2] Constructing line-level identifiers, quality flags & sparse-provider indicators...")

    # Unique Line Identifier
    if cols['line_num'] is not None and cols['line_num'] in df.columns:
        df['CLAIM_LINE_ID'] = df[cols['claim_id']].astype(str) + "_" + df[cols['line_num']].astype(str)
    else:
        df['CLAIM_LINE_ID'] = df[cols['claim_id']].astype(str) + "_L" + (df.index + 1).astype(str)

    # Handle duplicates if present in raw IDs
    if df['CLAIM_LINE_ID'].duplicated().any():
        df['CLAIM_LINE_ID'] = df['CLAIM_LINE_ID'] + "_IDX" + df.index.astype(str)

    # Data Quality Flags
    df['HAS_VALID_HCPCS'] = df['ORIGINAL_HCPCS_CD'].notna() & (df['ORIGINAL_HCPCS_CD'] != '') & (df['ORIGINAL_HCPCS_CD'] != 'nan')
    df['HAS_VALID_PROVIDER'] = df['ORIGINAL_PROVIDER_NPI'].notna() & (df['ORIGINAL_PROVIDER_NPI'] != '') & (df['ORIGINAL_PROVIDER_NPI'] != 'nan')
    df['HAS_VALID_DATE'] = df['ORIGINAL_FROM_DT'].notna()
    df['HAS_POSITIVE_PAYMENT'] = df['ORIGINAL_PMT_AMT'] > 0

    # Clean Analysis Eligibility Flag
    df['CLEAN_ANALYSIS_ELIGIBLE'] = (
        df['HAS_VALID_HCPCS'] &
        df['HAS_VALID_PROVIDER'] &
        df['HAS_VALID_DATE'] &
        (df['ORIGINAL_PMT_AMT'] >= 0)
    )

    # Sparse Provider Identification (< 3 claim lines)
    # Low-volume (sparse) providers have limited individual behavioral history and therefore rely more heavily on peer benchmarks.
    prov_line_counts = df.groupby('ORIGINAL_PROVIDER_NPI')['CLAIM_LINE_ID'].transform('count')
    df['IS_SPARSE_PROVIDER'] = prov_line_counts < 3

    print("\n--- DATA QUALITY & POPULATION SUMMARY ---")
    print(f"Total Claim Lines:           {len(df):,}")
    print(f"Valid HCPCS Lines:           {df['HAS_VALID_HCPCS'].sum():,}")
    print(f"Valid Provider Lines:        {df['HAS_VALID_PROVIDER'].sum():,}")
    print(f"Positive Payment Lines:      {df['HAS_POSITIVE_PAYMENT'].sum():,}")
    print(f"Zero Payment Lines:          {(df['ORIGINAL_PMT_AMT'] == 0).sum():,}")
    print(f"Missing Date Lines:          {(~df['HAS_VALID_DATE']).sum():,}")
    print(f"Duplicate CLAIM_LINE_IDs:    {df['CLAIM_LINE_ID'].duplicated().sum():,}")
    print(f"Sparse Provider Lines (<3):  {df['IS_SPARSE_PROVIDER'].sum():,} ({df['IS_SPARSE_PROVIDER'].mean()*100:.1f}%)")

    # =========================================================================
    # TASK 1 & 2 (CONT.): PROVIDER FEATURES & FIXED K=3 CLUSTERING
    # =========================================================================
    print("\n[Task 1 & 2] Building provider features & fitting K-Means (k=3)...")

    clean_records = df[df['CLEAN_ANALYSIS_ELIGIBLE']]
    payment_p75 = clean_records['ORIGINAL_PMT_AMT'].quantile(0.75)

    prov_features = clean_records.groupby('ORIGINAL_PROVIDER_NPI').agg(
        prov_median_pmt=('ORIGINAL_PMT_AMT', 'median'),
        prov_mean_pmt=('ORIGINAL_PMT_AMT', 'mean'),
        prov_high_pmt_share=('ORIGINAL_PMT_AMT', lambda x: (x > payment_p75).mean()),
        prov_total_lines=('CLAIM_LINE_ID', 'count'),
        prov_hcpcs_diversity=('ORIGINAL_HCPCS_CD', 'nunique')
    ).reset_index()

    # Log transformations for heavily skewed positive features
    prov_features['log_prov_mean_pmt'] = np.log1p(prov_features['prov_mean_pmt'].clip(lower=0))
    prov_features['log_prov_total_lines'] = np.log1p(prov_features['prov_total_lines'].clip(lower=0))
    prov_features['log_prov_hcpcs_diversity'] = np.log1p(prov_features['prov_hcpcs_diversity'].clip(lower=0))

    feature_cols_clustering = ['log_prov_mean_pmt', 'log_prov_total_lines', 'log_prov_hcpcs_diversity', 'prov_high_pmt_share']
    for c in feature_cols_clustering:
        prov_features[c] = prov_features[c].replace([np.inf, -np.inf], np.nan).fillna(prov_features[c].median())

    X_prov = prov_features[feature_cols_clustering].values
    scaler = StandardScaler()
    X_prov_scaled = scaler.fit_transform(X_prov)

    # Calculate Silhouette Scores for k=2..6 for documentation
    print("Calculating silhouette scores for documentation:")
    for k in range(2, 7):
        if len(prov_features) > k:
            km = KMeans(n_clusters=k, random_state=42, n_init=10)
            score = silhouette_score(X_prov_scaled, km.fit_predict(X_prov_scaled))
            print(f"  k={k}: Silhouette Score = {score:.4f}")

    # Explicitly select k=3 for business interpretability and adequate peer-group support
    print("INFO: Selected k=3 provider billing-behavior segments for interpretability and adequate peer-group support.")

    kmeans_3 = KMeans(n_clusters=3, random_state=42, n_init=10)
    prov_features['kmeans_cluster'] = kmeans_3.fit_predict(X_prov_scaled)

    # Map cluster assignments back to claim lines
    cluster_map = prov_features.set_index('ORIGINAL_PROVIDER_NPI')['kmeans_cluster'].to_dict()
    df['kmeans_cluster'] = df['ORIGINAL_PROVIDER_NPI'].map(cluster_map).fillna(0).astype(int)

    # Generate Cluster Profile Table (Provider Billing-Behavior Segments)
    profile_rows = []
    for cid in sorted(df['kmeans_cluster'].unique()):
        c_lines = df[df['kmeans_cluster'] == cid]
        c_provs = c_lines['ORIGINAL_PROVIDER_NPI'].unique()
        sparse_prov_cnt = (c_lines.groupby('ORIGINAL_PROVIDER_NPI')['CLAIM_LINE_ID'].count() < 3).sum()
        total_prov_cnt = len(c_provs)

        profile_rows.append({
            'Cluster ID': f"Segment {cid}",
            'Unique Provider Count': total_prov_cnt,
            'Sparse Provider Count': sparse_prov_cnt,
            'Sparse Provider Share (%)': f"{(sparse_prov_cnt / total_prov_cnt * 100):.1f}%" if total_prov_cnt > 0 else "0.0%",
            'Claim-Line Count': len(c_lines),
            'Median Payment ($)': round(c_lines['ORIGINAL_PMT_AMT'].median(), 2),
            'Mean Payment ($)': round(c_lines['ORIGINAL_PMT_AMT'].mean(), 2),
            'HCPCS Diversity (Avg)': round(c_lines.groupby('ORIGINAL_PROVIDER_NPI')['ORIGINAL_HCPCS_CD'].nunique().mean(), 1),
            'Total Volume Share (%)': f"{(len(c_lines)/len(df))*100:.1f}%"
        })
    provider_cluster_profile_df = pd.DataFrame(profile_rows)
    print("\n=== PROVIDER BILLING-BEHAVIOR SEGMENTS PROFILE ===")
    print(provider_cluster_profile_df.to_string(index=False))

    # =========================================================================
    # TASK 3: REFINED HIERARCHICAL EXPECTED-PAYMENT BENCHMARKS
    # =========================================================================
    print("\n[Task 3] Calculating hierarchical peer-group payment benchmarks...")

    # Calculate statistics on clean original positive-payment claims
    clean_pos_df = df[df['CLEAN_ANALYSIS_ELIGIBLE'] & df['HAS_POSITIVE_PAYMENT']].copy()

    # 1. Cluster + HCPCS statistics
    cl_hcpcs = clean_pos_df.groupby(['kmeans_cluster', 'ORIGINAL_HCPCS_CD'])['ORIGINAL_PMT_AMT'].agg(
        cluster_hcpcs_count='count',
        cluster_hcpcs_median_payment='median',
        cluster_hcpcs_mean_payment='mean',
        cluster_hcpcs_p95_pmt=lambda x: x.quantile(0.95),
        cluster_hcpcs_iqr=lambda x: x.quantile(0.75) - x.quantile(0.25)
    ).reset_index()

    # 2. HCPCS Global statistics
    gl_hcpcs = clean_pos_df.groupby('ORIGINAL_HCPCS_CD')['ORIGINAL_PMT_AMT'].agg(
        hcpcs_count='count',
        hcpcs_median_payment='median'
    ).reset_index()

    # 3. Cluster Global statistics
    cl_gl = clean_pos_df.groupby('kmeans_cluster')['ORIGINAL_PMT_AMT'].agg(
        cluster_count='count',
        cluster_median_payment='median'
    ).reset_index()

    # 4. Global Median Payment
    global_median_payment = clean_pos_df['ORIGINAL_PMT_AMT'].median()

    # Merge statistics onto main dataframe
    df = df.merge(cl_hcpcs, on=['kmeans_cluster', 'ORIGINAL_HCPCS_CD'], how='left')
    df = df.merge(gl_hcpcs, on='ORIGINAL_HCPCS_CD', how='left')
    df = df.merge(cl_gl, on='kmeans_cluster', how='left')

    # Apply Refined Fallback Hierarchy
    def assign_peer_benchmark(row):
        if row['cluster_hcpcs_count'] >= 5:
            return pd.Series([
                row['cluster_hcpcs_median_payment'],
                'cluster_hcpcs',
                row['cluster_hcpcs_count'],
                row['cluster_hcpcs_p95_pmt'],
                row['cluster_hcpcs_iqr']
            ])
        elif row['hcpcs_count'] >= 10:
            return pd.Series([
                row['hcpcs_median_payment'],
                'hcpcs_only',
                row['hcpcs_count'],
                np.nan,
                np.nan
            ])
        elif row['cluster_count'] >= 30:
            return pd.Series([
                row['cluster_median_payment'],
                'cluster_only',
                row['cluster_count'],
                np.nan,
                np.nan
            ])
        else:
            return pd.Series([
                global_median_payment,
                'global_fallback',
                len(clean_pos_df),
                np.nan,
                np.nan
            ])

    benchmark_cols = ['PEER_EXPECTED_PMT', 'PEER_GROUP_LEVEL', 'PEER_GROUP_COUNT', 'PEER_P95_PMT', 'PEER_PAYMENT_IQR']
    df[benchmark_cols] = df.apply(assign_peer_benchmark, axis=1)

    peer_summary_df = df['PEER_GROUP_LEVEL'].value_counts().reset_index()
    peer_summary_df.columns = ['Peer Group Level', 'Claim Lines']
    peer_summary_df['Percentage (%)'] = (peer_summary_df['Claim Lines'] / len(df) * 100).round(2)
    print("\n=== PEER BENCHMARK HIERARCHY COVERAGE ===")
    print(peer_summary_df.to_string(index=False))

    # =========================================================================
    # TASK 4 & 6: SCENARIO ELIGIBILITY & FEASIBLE ALLOCATION
    # =========================================================================
    print("\n[Task 4 & 6] Evaluating scenario eligibility flags & checking weights...")

    bene_col = cols['bene_id'] if cols['bene_id'] in df.columns else 'ORIGINAL_PROVIDER_NPI'

    df['ELIGIBLE_PRICE_DEVIATION'] = (
        (df['ORIGINAL_PMT_AMT'] > 0) &
        (df['PEER_EXPECTED_PMT'] > 0) &
        df['CLEAN_ANALYSIS_ELIGIBLE'] &
        df['PEER_GROUP_LEVEL'].notna()
    )

    df['ELIGIBLE_MODERATE_PAYMENT'] = (
        (df['ORIGINAL_PMT_AMT'] > 0) &
        (df['PEER_EXPECTED_PMT'] > 0) &
        df['CLEAN_ANALYSIS_ELIGIBLE'] &
        df['PEER_GROUP_LEVEL'].notna()
    )

    df['ELIGIBLE_DUPLICATE'] = (
        df['CLEAN_ANALYSIS_ELIGIBLE'] &
        df[bene_col].notna()
    )

    # Validate weights for 3 feasible scenarios
    scenario_weights = {
        "extreme_payment_deviation": 0.45,
        "duplicate_like_billing": 0.30,
        "moderate_payment_deviation": 0.25
    }
    assert np.isclose(sum(scenario_weights.values()), 1.0), "Scenario allocation weights must sum to 1.0"

    # =========================================================================
    # TASK 5, 7, 8, 9: PROPORTIONAL STRATIFIED ANOMALY INJECTION
    # =========================================================================
    print("\n[Task 5, 7, 8, 9] Executing proportional stratified anomaly injection engine...")

    rng = np.random.default_rng(42)

    df['IS_ANOMALY_INJECTED'] = 0
    df['SCENARIO_TYPE'] = "clean"
    df['INJECTION_MULTIPLIER'] = np.nan
    df['INJECTION_SEED'] = 42
    df['SOURCE_CLAIM_LINE_ID'] = df['CLAIM_LINE_ID']
    df['SYNTHETIC_RECORD_CREATED'] = 0

    target_rate = 0.035
    total_target_injections = int(np.round(len(df) * target_rate))

    allocated_indices = set()
    synthetic_duplicate_rows = []

    # Proportional Allocation per Scenario across Clusters
    for scenario_name, weight in scenario_weights.items():
        scenario_target_count = int(np.round(total_target_injections * weight))

        if scenario_name == 'extreme_payment_deviation':
            eligible_mask = df['ELIGIBLE_PRICE_DEVIATION']
        elif scenario_name == 'moderate_payment_deviation':
            eligible_mask = df['ELIGIBLE_MODERATE_PAYMENT']
        elif scenario_name == 'duplicate_like_billing':
            eligible_mask = df['ELIGIBLE_DUPLICATE']

        # Determine eligible count per cluster among unallocated original claim lines
        candidate_pool = df[eligible_mask & (~df.index.isin(allocated_indices))]
        cluster_eligible_counts = candidate_pool.groupby('kmeans_cluster').size().to_dict()

        # Ensure all clusters are present in dictionary
        for cid in df['kmeans_cluster'].unique():
            if cid not in cluster_eligible_counts:
                cluster_eligible_counts[cid] = 0

        # Calculate exact proportional allocation per cluster
        cluster_allocations = largest_remainder_allocation(scenario_target_count, cluster_eligible_counts)

        # Sample within each cluster
        selected_indices = []
        for cid, alloc_count in cluster_allocations.items():
            if alloc_count > 0:
                c_candidates = candidate_pool[candidate_pool['kmeans_cluster'] == cid].index.values
                sampled = rng.choice(c_candidates, size=alloc_count, replace=False).tolist()
                selected_indices.extend(sampled)

        # Track allocated claim lines for ALL scenarios to ensure non-overlapping selections
        allocated_indices.update(selected_indices)

        # Apply specific scenario modifications
        if scenario_name == 'extreme_payment_deviation':
            multipliers = rng.uniform(2.5, 4.0, size=len(selected_indices))
            for idx, mult in zip(selected_indices, multipliers):
                df.loc[idx, 'WORKING_PMT_AMT'] = df.loc[idx, 'PEER_EXPECTED_PMT'] * mult
                df.loc[idx, 'IS_ANOMALY_INJECTED'] = 1
                df.loc[idx, 'SCENARIO_TYPE'] = 'extreme_payment_deviation'
                df.loc[idx, 'INJECTION_MULTIPLIER'] = mult

        elif scenario_name == 'moderate_payment_deviation':
            multipliers = rng.uniform(1.4, 1.9, size=len(selected_indices))
            for idx, mult in zip(selected_indices, multipliers):
                df.loc[idx, 'WORKING_PMT_AMT'] = df.loc[idx, 'PEER_EXPECTED_PMT'] * mult
                df.loc[idx, 'IS_ANOMALY_INJECTED'] = 1
                df.loc[idx, 'SCENARIO_TYPE'] = 'moderate_payment_deviation'
                df.loc[idx, 'INJECTION_MULTIPLIER'] = mult

        elif scenario_name == 'duplicate_like_billing':
            for dup_count, idx in enumerate(selected_indices):
                source_row = df.loc[idx].copy()
                dup_row = source_row.copy()

                # New unique claim line identifier
                dup_row['CLAIM_LINE_ID'] = f"{source_row['CLAIM_LINE_ID']}_DUP{dup_count+1}"
                dup_row['SOURCE_CLAIM_LINE_ID'] = source_row['CLAIM_LINE_ID']

                # Service date offset (0 to 7 days)
                days_offset = rng.integers(0, 8)
                dup_row['WORKING_FROM_DT'] = source_row['ORIGINAL_FROM_DT'] + pd.Timedelta(days=int(days_offset))

                dup_row['IS_ANOMALY_INJECTED'] = 1
                dup_row['SCENARIO_TYPE'] = 'duplicate_like_billing'
                dup_row['SYNTHETIC_RECORD_CREATED'] = 1

                synthetic_duplicate_rows.append(dup_row)

    # Append synthetic duplicate rows to dataset
    if synthetic_duplicate_rows:
        synth_df = pd.DataFrame(synthetic_duplicate_rows)
        df = pd.concat([df, synth_df], ignore_index=True)

    print(f"INFO: Successfully injected synthetic scenarios across {df['IS_ANOMALY_INJECTED'].sum():,} total records.")

    # =========================================================================
    # TASK 10: RECALCULATE POST-INJECTION ANALYTICAL SIGNALS
    # =========================================================================
    print("\n[Task 10] Calculating post-injection analytical signals and ratios...")

    df['PAYMENT_RESIDUAL'] = df['WORKING_PMT_AMT'] - df['PEER_EXPECTED_PMT']
    df['PAYMENT_RATIO_TO_PEER'] = np.where(df['PEER_EXPECTED_PMT'] > 0, df['WORKING_PMT_AMT'] / df['PEER_EXPECTED_PMT'], np.nan)
    df['PAYMENT_DEVIATION_PERCENT'] = np.where(df['PEER_EXPECTED_PMT'] > 0, ((df['WORKING_PMT_AMT'] - df['PEER_EXPECTED_PMT']) / df['PEER_EXPECTED_PMT']) * 100.0, np.nan)

    # Candidate duplicate flag (beneficiary + provider + HCPCS + date window)
    dup_keys = [bene_col, 'WORKING_PROVIDER_NPI', 'WORKING_HCPCS_CD']
    df['DUPLICATE_CANDIDATE_FLAG'] = df.duplicated(subset=dup_keys, keep=False).astype(int)

    # Provider-HCPCS concentration feature
    prov_hcpcs_counts = df.groupby(['WORKING_PROVIDER_NPI', 'WORKING_HCPCS_CD'])['CLAIM_LINE_ID'].transform('count')
    prov_tot_counts = df.groupby('WORKING_PROVIDER_NPI')['CLAIM_LINE_ID'].transform('count')
    df['PROVIDER_HCPCS_CONCENTRATION'] = prov_hcpcs_counts / prov_tot_counts

    # =========================================================================
    # TASK 11: VALIDATION REPORT & ASSERTIONS
    # =========================================================================
    print("\n[Task 11] Running comprehensive validation report & safety assertions...")

    orig_len = len(merged_cms_df)
    final_len = len(df)
    total_inj = df['IS_ANOMALY_INJECTED'].sum()

    print(f"\n1. Population Count: Original={orig_len:,}, Final={final_len:,} (Synthetic Duplicates Added={final_len - orig_len:,})")
    print(f"2. Total Anomaly Rate vs Original Population: {total_inj / orig_len * 100:.2f}% ({total_inj:,} / {orig_len:,})")
    print(f"   Total Anomaly Rate vs Final Population:    {total_inj / final_len * 100:.2f}% ({total_inj:,} / {final_len:,})")

    # 3. Injection Count by Scenario
    print("\n3. Injection Count by Scenario:")
    scenario_report = df['SCENARIO_TYPE'].value_counts().reset_index()
    scenario_report.columns = ['Scenario Type', 'Count']
    print(scenario_report.to_string(index=False))

    # 4. Proportional Injection Rate by Cluster
    print("\n4. Cluster Injection Validation Table:")
    cluster_val = []
    for cid in sorted(df['kmeans_cluster'].unique()):
        sub_all = df[df['kmeans_cluster'] == cid]
        sub_orig = df[(df['kmeans_cluster'] == cid) & (df['SYNTHETIC_RECORD_CREATED'] == 0)]

        ext_cnt = (sub_all['SCENARIO_TYPE'] == 'extreme_payment_deviation').sum()
        mod_cnt = (sub_all['SCENARIO_TYPE'] == 'moderate_payment_deviation').sum()
        dup_cnt = (sub_all['SCENARIO_TYPE'] == 'duplicate_like_billing').sum()
        tot_inj_c = sub_all['IS_ANOMALY_INJECTED'].sum()

        cluster_val.append({
            'Cluster ID': f"Segment {cid}",
            'Total Records': len(sub_all),
            'Eligible Records': sub_orig['ELIGIBLE_PRICE_DEVIATION'].sum(),
            'Injected Records': tot_inj_c,
            'Injection Rate (%)': f"{(tot_inj_c / len(sub_all) * 100):.2f}%",
            'Extreme Price': ext_cnt,
            'Moderate Price': mod_cnt,
            'Duplicate': dup_cnt
        })
    cluster_val_df = pd.DataFrame(cluster_val)
    print(cluster_val_df.to_string(index=False))

    # 5. Injection Rate by Peer Group Level
    print("\n5. Injection Rate by Peer Group Level:")
    peer_val = df.groupby('PEER_GROUP_LEVEL')['IS_ANOMALY_INJECTED'].agg(Total='count', Injected='sum').reset_index()
    peer_val['Injection Rate (%)'] = (peer_val['Injected'] / peer_val['Total'] * 100).round(2)
    print(peer_val.to_string(index=False))

    # 6. Median Original vs Working Payment by Scenario
    print("\n6. Median Original vs Working Payment by Scenario:")
    pmt_val = df.groupby('SCENARIO_TYPE')[['ORIGINAL_PMT_AMT', 'WORKING_PMT_AMT']].median().reset_index()
    print(pmt_val.to_string(index=False))

    # Zero Payment & Overlap Validation Checks
    zero_pmt_injected = df[df['SCENARIO_TYPE'].isin(['extreme_payment_deviation', 'moderate_payment_deviation']) & (df['ORIGINAL_PMT_AMT'] == 0)].shape[0]
    overlapping_scenarios = (df.groupby('SOURCE_CLAIM_LINE_ID')['SCENARIO_TYPE'].transform(lambda x: (x != 'clean').sum()) > 1).sum()
    duplicate_line_ids = df['CLAIM_LINE_ID'].duplicated().sum()
    global_fallback_pct = (df['PEER_GROUP_LEVEL'] == 'global_fallback').mean() * 100

    print(f"\n7. Zero-Payment Claims Injected into Price Scenarios: {zero_pmt_injected} (Expected: 0)")
    print(f"8. Original Lines Assigned Multiple Scenarios:         {overlapping_scenarios} (Expected: 0)")
    print(f"9. Duplicate CLAIM_LINE_IDs in Final Dataset:            {duplicate_line_ids} (Expected: 0)")
    print(f"10. Global-Fallback Benchmark Coverage Percentage:       {global_fallback_pct:.2f}%")

    # --- ASSERTIONS ---
    assert zero_pmt_injected == 0, "Assertion Error: Zero-payment claim was selected for price deviation!"
    assert overlapping_scenarios == 0, "Assertion Error: Claim line assigned multiple scenarios!"
    assert duplicate_line_ids == 0, "Assertion Error: Duplicate CLAIM_LINE_ID found!"
    assert set(df['SCENARIO_TYPE'].unique()).issubset({
        'clean', 'extreme_payment_deviation', 'duplicate_like_billing', 'moderate_payment_deviation'
    }), "Assertion Error: Unrecognized scenario type present!"

    print("\n>>> ALL PIPELINE VALIDATION ASSERTIONS PASSED SUCCESSFULLY! <<<")

    # =========================================================================
    # TASK 12: SAVE REQUIRED OUTPUTS
    # =========================================================================
    print("\n[Task 12] Exporting final injected analytical dataset...")

    os.makedirs("data/processed", exist_ok=True)

    parquet_path = "data/processed/cms_claims_injected.parquet"
    csv_path = "data/processed/cms_claims_injected.csv"

    # String conversion for datetime columns before parquet export
    export_df = df.copy()
    for dt_col in export_df.select_dtypes(include=['datetime64', 'datetime']).columns:
        export_df[dt_col] = export_df[dt_col].astype(str)

    export_df.to_parquet(parquet_path, index=False)
    export_df.to_csv(csv_path, index=False)

    print(f"\nSaved Output Dataset:")
    print(f"  - Parquet: {parquet_path}")
    print(f"  - CSV:     {csv_path}")

    return df


# Demonstration execution harness if run directly
if __name__ == "__main__":
    if 'merged_cms_df' not in locals():
        print("Creating synthetic CMS sample dataset for pipeline execution...")
        np.random.seed(42)
        n_samples = 3000

        mock_df = pd.DataFrame({
            'DESY_SORT_KEY': [f"BENE_{np.random.randint(1000, 2000)}" for _ in range(n_samples)],
            'CLM_ID': [f"CLM_{10000 + i}" for i in range(n_samples)],
            'LINE_NUM': np.random.randint(1, 4, size=n_samples),
            'PRF_PHYSN_NPI_1': [f"NPI_{np.random.randint(100, 250)}" for _ in range(n_samples)],
            'HCPCS_CD_1': np.random.choice(['99213', '99214', '71045', '93000', '36415'], size=n_samples),
            'CLM_FROM_DT': pd.date_range(start='2025-01-01', periods=n_samples, freq='h'),
            'LINE_NCH_PMT_AMT_1': np.random.choice([0.0, 45.0, 85.0, 150.0, 320.0], size=n_samples, p=[0.05, 0.40, 0.30, 0.20, 0.05]),
            'PROVIDER_SPECIALTY': np.random.choice(['Radiology', 'Cardiology', 'General Practice'], size=n_samples)
        })
        merged_cms_df = mock_df

    processed_df = run_pipeline(merged_cms_df)

STARTING SYNTHETIC PAYMENT INTEGRITY DATA PROCESSING & SCENARIO INJECTION

[Task 1] Preserving immutable original baseline columns & working fields...
[Task 2] Constructing line-level identifiers, quality flags & sparse-provider indicators...

--- DATA QUALITY & POPULATION SUMMARY ---
Total Claim Lines:           10,000
Valid HCPCS Lines:           9,909
Valid Provider Lines:        9,976
Positive Payment Lines:      8,312
Zero Payment Lines:          1,688
Missing Date Lines:          0
Duplicate CLAIM_LINE_IDs:    0
Sparse Provider Lines (<3):  8,163 (81.6%)

[Task 1 & 2] Building provider features & fitting K-Means (k=3)...
Calculating silhouette scores for documentation:
  k=2: Silhouette Score = 0.5998
  k=3: Silhouette Score = 0.6316
  k=4: Silhouette Score = 0.6874
  k=5: Silhouette Score = 0.7869
  k=6: Silhouette Score = 0.7922
INFO: Selected k=3 provider billing-behavior segments for interpretability and adequate peer-group support.

=== PROVIDER BILLING-BEHAVIOR SEGMENTS PRO

# Stage 3: Predictive Baseline Model (Predict Expected Payment based on Clinical Features)

In [9]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Setup Directories
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)

INPUT_PATH = "data/processed/cms_claims_injected.parquet"

if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(
        f"Stage 2 input not found: {INPUT_PATH}. "
        "Run Stage 2 before Stage 3."
    )

import pandas as pd
import numpy as np

def calculate_clinical_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derives PATIENT_AGE, AGE_MISSING_FLAG, and CHRONIC_CONDITION_COUNT
    from standard CMS claims fields.
    """
    df = df.copy()

    # -------------------------------------------------------------------------
    # 1. PATIENT_AGE & AGE_MISSING_FLAG
    # -------------------------------------------------------------------------
    # Auto-detect standard CMS birth date columns
    birth_col = next((c for c in ['BENE_BIRTH_DT', 'BENE_BIRTH_DATE', 'DOB', 'PATIENT_DOB', 'BIRTH_DT'] if c in df.columns), None)

    # Auto-detect standard CMS claim / service date columns
    claim_date_col = next((c for c in ['CLM_FROM_DT', 'CLM_THRU_DT', 'LINE_1ST_EXPNS_DT', 'SRVC_DT', 'CLAIM_DATE'] if c in df.columns), None)

    if "PATIENT_AGE" not in df.columns or df["PATIENT_AGE"].isna().all():
        if birth_col and claim_date_col:
            birth_dt = pd.to_datetime(df[birth_col], errors='coerce')
            claim_dt = pd.to_datetime(df[claim_date_col], errors='coerce')

            # Calculate age at time of service
            df["PATIENT_AGE"] = (claim_dt - birth_dt).dt.days / 365.25
        else:
            df["PATIENT_AGE"] = np.nan

    # Identify missing or out-of-range ages (< 0 or > 120)
    invalid_age_mask = df["PATIENT_AGE"].isna() | (df["PATIENT_AGE"] < 0) | (df["PATIENT_AGE"] > 120)
    df["AGE_MISSING_FLAG"] = invalid_age_mask.astype(int)

    # Impute missing ages using the median age (default to 65.0 Medicare baseline if median is empty)
    median_age = df.loc[~invalid_age_mask, "PATIENT_AGE"].median()
    if pd.isna(median_age):
        median_age = 65.0

    df["PATIENT_AGE"] = df["PATIENT_AGE"].where(~invalid_age_mask, median_age)

    # -------------------------------------------------------------------------
    # 2. CHRONIC_CONDITION_COUNT
    # -------------------------------------------------------------------------
    if "CHRONIC_CONDITION_COUNT" not in df.columns or df["CHRONIC_CONDITION_COUNT"].isna().all():
        # Look for standard CMS SynPUF condition columns (e.g., SP_ALZHMR, SP_CHF, SP_DIABETES, SP_CRDHYP)
        sp_cols = [c for c in df.columns if c.startswith("SP_") or "CHRONIC" in c.upper()]

        if sp_cols:
            # In CMS DE-SynPUF: Code 1 indicates 'Yes' for the condition (Code 2 is 'No')
            df["CHRONIC_CONDITION_COUNT"] = (df[sp_cols] == 1).sum(axis=1)
        else:
            # Fallback: Count populated ICD diagnosis code fields if SP_ columns are absent
            diag_cols = [c for c in df.columns if "ICD_DGNS" in c or "DIAG" in c]
            if diag_cols:
                df["CHRONIC_CONDITION_COUNT"] = df[diag_cols].notna().sum(axis=1)
            else:
                df["CHRONIC_CONDITION_COUNT"] = 0

    return df


# Load data
df = pd.read_parquet(INPUT_PATH)
print(f"INFO: Loaded Dataset. Total Rows: {len(df)}")

# =====================================================================
# 1. PRESERVE THE CLEAN GROUPED SPLIT
# =====================================================================

original_record_mask = (df["SYNTHETIC_RECORD_CREATED"] == 0)
clean_original_mask = (
    original_record_mask &
    (df["IS_ANOMALY_INJECTED"] == 0) &
    df["ORIGINAL_PMT_AMT"].notna()
)

# Calculate missing clinical features
df = calculate_clinical_features(df)

print("Derived features successfully:")
# print(df[['PATIENT_AGE', 'AGE_MISSING_FLAG', 'CHRONIC_CONDITION_COUNT']].head())

clean_df = df[clean_original_mask].copy()
group_col = 'CLM_ID' if 'CLM_ID' in clean_df.columns else clean_df.index
groups = clean_df[group_col] if 'CLM_ID' in clean_df.columns else clean_df.index.values

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx_loc, holdout_idx_loc = next(gss.split(clean_df, groups=groups))

train_clean_indices = clean_df.iloc[train_idx_loc].index
holdout_clean_indices = clean_df.iloc[holdout_idx_loc].index

clean_train_mask = df.index.isin(train_clean_indices)
clean_holdout_mask = df.index.isin(holdout_clean_indices)

# Validate group overlap
if 'CLM_ID' in clean_df.columns:
    train_groups = set(df[clean_train_mask]['CLM_ID'])
    holdout_groups = set(df[clean_holdout_mask]['CLM_ID'])
    assert len(train_groups.intersection(holdout_groups)) == 0, "Group overlap detected!"

print(f"INFO: Clean Train Records: {clean_train_mask.sum()}, Clean Holdout Records: {clean_holdout_mask.sum()}")

# =====================================================================
# 2. CREATE A POSITIVE-PAYMENT EXPECTED-PAYMENT POPULATION
# =====================================================================
positive_payment_train_mask = clean_train_mask & (df['ORIGINAL_PMT_AMT'] > 0)
positive_payment_holdout_mask = clean_holdout_mask & (df['ORIGINAL_PMT_AMT'] > 0)

print(f"INFO: Positive-Payment Train Records: {positive_payment_train_mask.sum()}, Holdout: {positive_payment_holdout_mask.sum()}")

# Peer Benchmark calculation function
def compute_leakage_free_peer_benchmarks(train_source, target_df):
    cl_hcpcs = train_source.groupby(['kmeans_cluster', 'ORIGINAL_HCPCS_CD'])['ORIGINAL_PMT_AMT'].agg(
        tr_cl_hcpcs_count='count', tr_cl_hcpcs_median='median').reset_index()
    gl_hcpcs = train_source.groupby('ORIGINAL_HCPCS_CD')['ORIGINAL_PMT_AMT'].agg(
        tr_hcpcs_count='count', tr_hcpcs_median='median').reset_index()
    cl_gl = train_source.groupby('kmeans_cluster')['ORIGINAL_PMT_AMT'].agg(
        tr_cl_count='count', tr_cl_median='median').reset_index()

    global_median = train_source['ORIGINAL_PMT_AMT'].median()

    mapped_df = target_df.copy()
    mapped_df = mapped_df.merge(cl_hcpcs, on=['kmeans_cluster', 'ORIGINAL_HCPCS_CD'], how='left')
    mapped_df = mapped_df.merge(gl_hcpcs, on='ORIGINAL_HCPCS_CD', how='left')
    mapped_df = mapped_df.merge(cl_gl, on='kmeans_cluster', how='left')

    mapped_df['tr_cl_hcpcs_count'] = mapped_df['tr_cl_hcpcs_count'].fillna(0)
    mapped_df['tr_hcpcs_count'] = mapped_df['tr_hcpcs_count'].fillna(0)
    mapped_df['tr_cl_count'] = mapped_df['tr_cl_count'].fillna(0)

    conds = [
        mapped_df['tr_cl_hcpcs_count'] >= 5,
        mapped_df['tr_hcpcs_count'] >= 10,
        mapped_df['tr_cl_count'] >= 30
    ]

    mapped_df['MODEL_PEER_EXPECTED_PMT'] = np.select(
        conds,
        [mapped_df['tr_cl_hcpcs_median'], mapped_df['tr_hcpcs_median'], mapped_df['tr_cl_median']],
        default=global_median
    )
    mapped_df['MODEL_PEER_GROUP_LEVEL'] = np.select(
        conds, ['cluster_hcpcs', 'hcpcs_only', 'cluster_only'], default='global_fallback'
    )
    mapped_df['MODEL_PEER_GROUP_COUNT'] = np.select(
        conds, [mapped_df['tr_cl_hcpcs_count'], mapped_df['tr_hcpcs_count'], mapped_df['tr_cl_count']], default=len(train_source)
    )
    mapped_df['LOG_MODEL_PEER_EXPECTED_PMT'] = np.log1p(np.maximum(0.0, mapped_df['MODEL_PEER_EXPECTED_PMT']))

    mapped_df.drop(columns=['tr_cl_hcpcs_count', 'tr_cl_hcpcs_median', 'tr_hcpcs_count', 'tr_hcpcs_median', 'tr_cl_count', 'tr_cl_median'], inplace=True)
    return mapped_df

# =====================================================================
# 3. COMPARE THREE BASELINES ON CLEAN POSITIVE-PAYMENT HOLDOUT
# =====================================================================
train_pos_df = df[positive_payment_train_mask].copy()
holdout_pos_df = df[positive_payment_holdout_mask].copy()

# Compute peer benchmarks strictly from positive-payment training data
train_pos_df = compute_leakage_free_peer_benchmarks(train_pos_df, train_pos_df)
holdout_pos_df = compute_leakage_free_peer_benchmarks(train_pos_df, holdout_pos_df)

numeric_features = ['PATIENT_AGE', 'AGE_MISSING_FLAG', 'CHRONIC_CONDITION_COUNT', 'LOG_MODEL_PEER_EXPECTED_PMT', 'MODEL_PEER_GROUP_COUNT']
candidate_cat = ['ORIGINAL_HCPCS_CD', 'kmeans_cluster', 'MODEL_PEER_GROUP_LEVEL']
categorical_features = [c for c in candidate_cat if c in train_pos_df.columns]

for c in categorical_features:
    train_pos_df[c] = train_pos_df[c].astype(str)
    holdout_pos_df[c] = holdout_pos_df[c].astype(str)

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features)
])

# Baseline A: Peer
y_holdout = holdout_pos_df['ORIGINAL_PMT_AMT'].values
pred_A = holdout_pos_df['MODEL_PEER_EXPECTED_PMT'].values

# Baseline B: Ridge Raw
X_train = train_pos_df[numeric_features + categorical_features]
y_train_raw = train_pos_df['ORIGINAL_PMT_AMT'].values
ridge_raw = Pipeline([('prep', preprocessor), ('model', Ridge(alpha=1.0, random_state=42))])
ridge_raw.fit(X_train, y_train_raw)
pred_B_raw = ridge_raw.predict(holdout_pos_df[numeric_features + categorical_features])
pred_B = np.maximum(0, pred_B_raw)

# Baseline C: Ridge Log + Smearing
y_train_log = np.log1p(y_train_raw)
ridge_log = Pipeline([('prep', preprocessor), ('model', Ridge(alpha=1.0, random_state=42))])
ridge_log.fit(X_train, y_train_log)

train_pred_log = ridge_log.predict(X_train)
train_log_residuals = y_train_log - train_pred_log
smearing_factor = np.mean(np.exp(train_log_residuals))
print(f"INFO: Calculated Duan Smearing Factor (Train Only): {smearing_factor:.4f}")

holdout_pred_log = ridge_log.predict(holdout_pos_df[numeric_features + categorical_features])
pred_C = np.maximum(0, (np.exp(holdout_pred_log) * smearing_factor) - 1)

def eval_metrics(actual, pred):
    wape = (np.sum(np.abs(actual - pred)) / np.sum(np.abs(actual))) * 100
    return {
        'MAE': mean_absolute_error(actual, pred),
        'Median AE': np.median(np.abs(actual - pred)),
        'RMSE': np.sqrt(mean_squared_error(actual, pred)),
        'R2': r2_score(actual, pred),
        'WAPE (%)': wape
    }

metrics_A = eval_metrics(y_holdout, pred_A)
metrics_B = eval_metrics(y_holdout, pred_B)
metrics_C = eval_metrics(y_holdout, pred_C)

print("\n--- BASELINE EVALUATION (CLEAN POSITIVE HOLDOUT) ---")
print(pd.DataFrame({'Peer benchmark': metrics_A, 'Ridge raw': metrics_B, 'Ridge log + smearing': metrics_C}).T)

# =====================================================================
# 4. SELECT THE BEST BASELINE
# =====================================================================
wape_vals = {'peer_benchmark': metrics_A['WAPE (%)'], 'ridge_raw': metrics_B['WAPE (%)'], 'ridge_log_smearing': metrics_C['WAPE (%)']}
min_wape = min(wape_vals.values())

# Selection logic (within 1% prefer simpler: Peer > Raw > Log)
if wape_vals['peer_benchmark'] <= min_wape + 1.0:
    SELECTED_EXPECTED_PMT_METHOD = 'peer_benchmark'
elif wape_vals['ridge_raw'] <= min_wape + 1.0:
    SELECTED_EXPECTED_PMT_METHOD = 'ridge_raw'
else:
    SELECTED_EXPECTED_PMT_METHOD = 'ridge_log_smearing'

print(f"\nINFO: Selected Expected-Payment Method: {SELECTED_EXPECTED_PMT_METHOD}")

# =====================================================================
# 5. REFIT ON ALL CLEAN POSITIVE-PAYMENT DATA
# =====================================================================
all_clean_pos_mask = clean_original_mask & (df['ORIGINAL_PMT_AMT'] > 0)
all_clean_pos_df = df[all_clean_pos_mask].copy()

# Rebuild peer mappings from ALL clean positive records
all_clean_pos_df = compute_leakage_free_peer_benchmarks(all_clean_pos_df, all_clean_pos_df)
df_scored = compute_leakage_free_peer_benchmarks(all_clean_pos_df, df.copy())

train_group_set = set(df.loc[train_clean_indices, "CLM_ID"])
holdout_group_set = set(df.loc[holdout_clean_indices, "CLM_ID"])

df_scored["MODEL_PARTITION"] = np.select(
    [
        df_scored["CLM_ID"].isin(train_group_set),
        df_scored["CLM_ID"].isin(holdout_group_set)
    ],
    [
        "train",
        "holdout"
    ],
    default="unassigned"
)

for c in categorical_features:
    all_clean_pos_df[c] = all_clean_pos_df[c].astype(str)
    df_scored[c] = df_scored[c].astype(str)

X_all = all_clean_pos_df[numeric_features + categorical_features]
y_all_raw = all_clean_pos_df['ORIGINAL_PMT_AMT'].values
y_all_log = np.log1p(y_all_raw)

ridge_raw.fit(X_all, y_all_raw)
ridge_log.fit(X_all, y_all_log)

all_train_pred_log = ridge_log.predict(X_all)
all_train_log_residuals = y_all_log - all_train_pred_log
final_smearing_factor = np.mean(np.exp(all_train_log_residuals))

X_full = df_scored[numeric_features + categorical_features]
df_scored['EXPECTED_PMT_PEER'] = df_scored['MODEL_PEER_EXPECTED_PMT']
df_scored['EXPECTED_PMT_RIDGE_RAW'] = np.maximum(0, ridge_raw.predict(X_full))
df_scored['EXPECTED_PMT_RIDGE_LOG_SMEAR'] = np.maximum(0, (np.exp(ridge_log.predict(X_full)) * final_smearing_factor) - 1)

if SELECTED_EXPECTED_PMT_METHOD == 'peer_benchmark':
    df_scored['EXPECTED_PMT_FINAL'] = df_scored['EXPECTED_PMT_PEER']
elif SELECTED_EXPECTED_PMT_METHOD == 'ridge_raw':
    df_scored['EXPECTED_PMT_FINAL'] = df_scored['EXPECTED_PMT_RIDGE_RAW']
else:
    df_scored['EXPECTED_PMT_FINAL'] = df_scored['EXPECTED_PMT_RIDGE_LOG_SMEAR']

# =====================================================================
# 6. CREATE CALIBRATED PRICE-DEVIATION SIGNALS
# =====================================================================
df_scored['PRICE_SIGNAL_ELIGIBLE'] = (df_scored['WORKING_PMT_AMT'] > 0) & (df_scored['EXPECTED_PMT_FINAL'] > 0) & df_scored['EXPECTED_PMT_FINAL'].notna()

df_scored['PAYMENT_RESIDUAL_FINAL'] = df_scored['WORKING_PMT_AMT'] - df_scored['EXPECTED_PMT_FINAL']
df_scored['ABS_PAYMENT_RESIDUAL_FINAL'] = np.abs(df_scored['PAYMENT_RESIDUAL_FINAL'])

df_scored['RELATIVE_SURGE_RATIO_FINAL'] = np.where(
    df_scored['PRICE_SIGNAL_ELIGIBLE'],
    (df_scored['WORKING_PMT_AMT'] - df_scored['EXPECTED_PMT_FINAL']) / df_scored['EXPECTED_PMT_FINAL'],
    np.nan
)
df_scored['PAYMENT_RATIO_TO_EXPECTED_FINAL'] = np.where(
    df_scored['PRICE_SIGNAL_ELIGIBLE'],
    df_scored['WORKING_PMT_AMT'] / df_scored['EXPECTED_PMT_FINAL'],
    np.nan
)
df_scored['EXPECTED_PMT_FINAL_ZERO_FLAG'] = (df_scored['EXPECTED_PMT_FINAL'] == 0).astype(int)

# =====================================================================
# 7. HOLDOUT-CALIBRATED THRESHOLDS & TIERING
# =====================================================================
# Use clean positive holdout records to find P95/P99 of surge ratio
holdout_surge_mask = positive_payment_holdout_mask & df_scored['PRICE_SIGNAL_ELIGIBLE']
holdout_surge_ratios = df_scored.loc[holdout_surge_mask, 'RELATIVE_SURGE_RATIO_FINAL'].dropna()

PAYMENT_SURGE_P95_THRESHOLD = np.percentile(holdout_surge_ratios, 95) if len(holdout_surge_ratios) > 0 else 1.0
PAYMENT_SURGE_P99_THRESHOLD = np.percentile(holdout_surge_ratios, 99) if len(holdout_surge_ratios) > 0 else 2.0

print(f"\nINFO: Holdout-Calibrated Thresholds -> P95: {PAYMENT_SURGE_P95_THRESHOLD:.3f}, P99: {PAYMENT_SURGE_P99_THRESHOLD:.3f}")

def assign_tier(row):
    if not row['PRICE_SIGNAL_ELIGIBLE'] or pd.isna(row['RELATIVE_SURGE_RATIO_FINAL']):
        return 'Not eligible'
    if row['RELATIVE_SURGE_RATIO_FINAL'] >= PAYMENT_SURGE_P99_THRESHOLD:
        return 'High'
    elif row['RELATIVE_SURGE_RATIO_FINAL'] >= PAYMENT_SURGE_P95_THRESHOLD:
        return 'Medium'
    else:
        return 'Standard'

df_scored['PAYMENT_DEVIATION_TIER'] = df_scored.apply(assign_tier, axis=1)

# =====================================================================
# 8. DIAGNOSTICS
# =====================================================================
print("\n=== A. Selected-Model Holdout Calibration ===")
actual_h = df_scored.loc[positive_payment_holdout_mask, 'ORIGINAL_PMT_AMT']
pred_h = df_scored.loc[positive_payment_holdout_mask, 'EXPECTED_PMT_FINAL']
resid_h = df_scored.loc[positive_payment_holdout_mask, 'PAYMENT_RESIDUAL_FINAL']
print(f"Actual mean payment:    ${actual_h.mean():.2f}")
print(f"Predicted mean payment: ${pred_h.mean():.2f}")
print(f"Actual median payment:  ${actual_h.median():.2f}")
print(f"Predicted median payment:${pred_h.median():.2f}")
print(f"Mean residual:          ${resid_h.mean():.2f}")
print(f"Median residual:        ${resid_h.median():.2f}")
print(f"MAE:                    ${mean_absolute_error(actual_h, pred_h):.2f}")
print(f"WAPE:                   {(np.sum(np.abs(actual_h - pred_h)) / np.sum(actual_h))*100:.2f}%")

print("\n=== C. Scenario Signal Diagnostics ===")
scenarios = ['clean', 'extreme_payment_deviation', 'moderate_payment_deviation', 'duplicate_like_billing']
for s in scenarios:
    s_df = df_scored[df_scored['SCENARIO_TYPE'] == s]
    p95_pct = (s_df['RELATIVE_SURGE_RATIO_FINAL'] >= PAYMENT_SURGE_P95_THRESHOLD).mean() * 100
    p99_pct = (s_df['RELATIVE_SURGE_RATIO_FINAL'] >= PAYMENT_SURGE_P99_THRESHOLD).mean() * 100
    elig_pct = s_df['PRICE_SIGNAL_ELIGIBLE'].mean() * 100

    print(f"- {s}: N={len(s_df)}")
    print(f"  Median Working Pmt: ${s_df['WORKING_PMT_AMT'].median():.2f}")
    print(f"  Median Expected Pmt: ${s_df['EXPECTED_PMT_FINAL'].median():.2f}")
    print(f"  Median Residual: ${s_df['PAYMENT_RESIDUAL_FINAL'].median():.2f}")
    print(f"  % Above P95: {p95_pct:.1f}%")
    print(f"  % Above P99: {p99_pct:.1f}%")
    print(f"  % Eligible: {elig_pct:.1f}%\n")

print("=== D. Zero-payment Diagnostics ===")
zero_mask = clean_original_mask & (df['ORIGINAL_PMT_AMT'] == 0)
zero_claims = df_scored[zero_mask]
print(f"Total zero-payment clean claims: {len(zero_claims)}")
print(f"Excluded from PRICE_SIGNAL_ELIGIBLE: {(~zero_claims['PRICE_SIGNAL_ELIGIBLE']).sum()}")
print(f"Undefined price ratio (NaN): {zero_claims['RELATIVE_SURGE_RATIO_FINAL'].isna().sum()}")
# Confirm tier assignment for zero-payment
assert (zero_claims['PAYMENT_DEVIATION_TIER'] == 'Not eligible').all(), "Zero-payment claims incorrectly tiered!"

# =====================================================================
# 10. ASSERTIONS
# =====================================================================
assert (df_scored.loc[clean_train_mask, 'IS_ANOMALY_INJECTED'] == 0).all()
assert (df_scored.loc[clean_train_mask, 'SYNTHETIC_RECORD_CREATED'] == 0).all()
assert len(train_groups.intersection(holdout_groups)) == 0
assert not df_scored.loc[df_scored['EXPECTED_PMT_FINAL'] == 0, 'PRICE_SIGNAL_ELIGIBLE'].any()
assert df_scored.loc[~df_scored['PRICE_SIGNAL_ELIGIBLE'], 'RELATIVE_SURGE_RATIO_FINAL'].isna().all()
assert (df_scored['EXPECTED_PMT_FINAL'] >= 0).all()

# Save output
PARQUET_OUT = 'data/processed/cms_claims_stage3_calibrated.parquet'
CSV_OUT = 'data/processed/cms_claims_stage3_calibrated.csv'

df_scored.to_parquet(PARQUET_OUT, index=False)
df_scored.to_csv(CSV_OUT, index=False)
joblib.dump(ridge_raw, 'models/ridge_expected_payment_raw_pipeline.joblib')
joblib.dump(ridge_log, 'models/ridge_expected_payment_log_smearing_pipeline.joblib')

print(f"\nSUCCESS. Saved artifacts:\n{PARQUET_OUT}\n{CSV_OUT}")

INFO: Loaded Dataset. Total Rows: 10105
Derived features successfully:
INFO: Clean Train Records: 7803, Clean Holdout Records: 1951
INFO: Positive-Payment Train Records: 6467, Holdout: 1599
INFO: Calculated Duan Smearing Factor (Train Only): 1.2388

--- BASELINE EVALUATION (CLEAN POSITIVE HOLDOUT) ---
                            MAE  Median AE       RMSE        R2   WAPE (%)
Peer benchmark        38.130081  20.000000  78.144324  0.201806  51.805591
Ridge raw             40.758575  24.128296  72.876267  0.305798  55.376804
Ridge log + smearing  40.347649  22.418537  74.091662  0.282450  54.818498

INFO: Selected Expected-Payment Method: peer_benchmark

INFO: Holdout-Calibrated Thresholds -> P95: 2.275, P99: 6.000

=== A. Selected-Model Holdout Calibration ===
Actual mean payment:    $73.60
Predicted mean payment: $58.70
Actual median payment:  $50.00
Predicted median payment:$50.00
Mean residual:          $14.91
Median residual:        $0.00
MAE:                    $36.16
WAPE:         

Level 1 (Transaction Engine): Evaluates single-claim price inflation ($Y_{\text{actual}} \text{ vs } Y_{\text{expected}}$).  
Level 2 (Behavioral Time-Series): Evaluates a provider's shift against their own history ($Z_{\text{rolling}}$).  
Level 3 (Cluster Isolation Forest): Evaluates multivariate peer-group outliers against similar-sized clinics.

#Stage 4: TRANSACTION-LEVEL ANOMALY ENGINE (UN-GROUPED CLAIM LEVEL)


In [10]:
"""
Stage 4: Transaction-Level Price-Deviation Anomaly Engine
(Colab Output & File Storage Enabled)
"""

import os
import logging
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def run_stage4_price_deviation_engine(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies Stage 4 Price-Deviation logic to CMS claims data.
    (Model logic and assertions preserved exactly as defined)
    """
    df = df.copy()

    logging.info("Starting Stage 4: Transaction-Level Price-Deviation Engine...")

    # ---------------------------------------------------------
    # 1. Cluster Key Normalization
    # ---------------------------------------------------------
    df["STAGE4_CLUSTER_KEY"] = df["kmeans_cluster"].astype(str).str.strip()

    # ---------------------------------------------------------
    # 2. Correct Zero-Payment & Ineligible Handling
    # ---------------------------------------------------------
    if "WORKING_PMT_AMT" not in df.columns:
        possible_pmt_cols = ["LINE_CVRD_PD_AMT", "PMT_AMT", "PAYMENT_AMOUNT"]
        for col in possible_pmt_cols:
            if col in df.columns:
                df["WORKING_PMT_AMT"] = df[col]
                break

    df["PRICE_SIGNAL_ELIGIBLE"] = (
        df["WORKING_PMT_AMT"].gt(0) &
        df["EXPECTED_PMT_FINAL"].gt(0) &
        df["RELATIVE_SURGE_RATIO_FINAL"].notna() &
        df["PAYMENT_RESIDUAL_FINAL"].notna()
    )

    # Initialize Stage 4 columns for all rows with "not_eligible" safe defaults
    df["STAGE4_POSITIVE_PAYMENT_RESIDUAL"] = np.nan
    df["STAGE4_RELATIVE_SURGE_PERCENTILE"] = np.nan
    df["STAGE4_POSITIVE_RESIDUAL_PERCENTILE"] = np.nan
    df["STAGE4_PRICE_RISK_SCORE"] = np.nan
    df["STAGE4_PRICE_TIER"] = "not_eligible"
    df["STAGE4_PRICE_FLAG"] = 0
    df["STAGE4_THRESHOLD_SOURCE"] = "not_eligible"

    df["STAGE4_APPLIED_SURGE_P95"] = np.nan
    df["STAGE4_APPLIED_SURGE_P99"] = np.nan
    df["STAGE4_APPLIED_RESIDUAL_P90"] = np.nan
    df["STAGE4_APPLIED_RESIDUAL_P95"] = np.nan

    # For eligible rows only: Calculate positive residual
    eligible_mask = df["PRICE_SIGNAL_ELIGIBLE"] == True
    df.loc[eligible_mask, "STAGE4_POSITIVE_PAYMENT_RESIDUAL"] = np.maximum(
        df.loc[eligible_mask, "PAYMENT_RESIDUAL_FINAL"],
        0
    )

# ---------------------------------------------------------
    # 3. Threshold Calibration
    # ---------------------------------------------------------
    # Safe mask construction in case split/injection metadata columns are missing
    syn_mask = (df["SYNTHETIC_RECORD_CREATED"] == 0) if "SYNTHETIC_RECORD_CREATED" in df.columns else True
    inj_mask = (df["IS_ANOMALY_INJECTED"] == 0) if "IS_ANOMALY_INJECTED" in df.columns else True
    part_mask = (df["MODEL_PARTITION"] == "holdout") if "MODEL_PARTITION" in df.columns else True

    clean_holdout_mask = (
        syn_mask &
        inj_mask &
        part_mask &
        (df["PRICE_SIGNAL_ELIGIBLE"] == True)
    )

    clean_holdout_df = df[clean_holdout_mask]

    # Calculate global fallback metrics
    global_surge_p95 = clean_holdout_df["RELATIVE_SURGE_RATIO_FINAL"].quantile(0.95)
    global_surge_p99 = clean_holdout_df["RELATIVE_SURGE_RATIO_FINAL"].quantile(0.99)
    global_res_p90 = clean_holdout_df["STAGE4_POSITIVE_PAYMENT_RESIDUAL"].quantile(0.90)
    global_res_p95 = clean_holdout_df["STAGE4_POSITIVE_PAYMENT_RESIDUAL"].quantile(0.95)

    # Extract arrays for efficient percentile calculations later
    global_surge_dist = np.sort(clean_holdout_df["RELATIVE_SURGE_RATIO_FINAL"].dropna().values)
    global_res_dist = np.sort(clean_holdout_df["STAGE4_POSITIVE_PAYMENT_RESIDUAL"].dropna().values)

    # Process metrics using standard cluster keys
    for cluster_key in df["STAGE4_CLUSTER_KEY"].unique():
        cluster_overall_mask = (df["STAGE4_CLUSTER_KEY"] == cluster_key) & eligible_mask
        cluster_clean_holdout = clean_holdout_df[clean_holdout_df["STAGE4_CLUSTER_KEY"] == cluster_key]

        if len(cluster_clean_holdout) >= 30:
            surge_p95 = cluster_clean_holdout["RELATIVE_SURGE_RATIO_FINAL"].quantile(0.95)
            surge_p99 = cluster_clean_holdout["RELATIVE_SURGE_RATIO_FINAL"].quantile(0.99)
            res_p90 = cluster_clean_holdout["STAGE4_POSITIVE_PAYMENT_RESIDUAL"].quantile(0.90)
            res_p95 = cluster_clean_holdout["STAGE4_POSITIVE_PAYMENT_RESIDUAL"].quantile(0.95)
            thresh_source = "cluster_holdout"

            surge_dist = np.sort(cluster_clean_holdout["RELATIVE_SURGE_RATIO_FINAL"].dropna().values)
            res_dist = np.sort(cluster_clean_holdout["STAGE4_POSITIVE_PAYMENT_RESIDUAL"].dropna().values)
        else:
            surge_p95 = global_surge_p95
            surge_p99 = global_surge_p99
            res_p90 = global_res_p90
            res_p95 = global_res_p95
            thresh_source = "global_holdout_fallback"

            surge_dist = global_surge_dist
            res_dist = global_res_dist

        # Store applied threshold values to the main dataframe
        df.loc[cluster_overall_mask, "STAGE4_APPLIED_SURGE_P95"] = surge_p95
        df.loc[cluster_overall_mask, "STAGE4_APPLIED_SURGE_P99"] = surge_p99
        df.loc[cluster_overall_mask, "STAGE4_APPLIED_RESIDUAL_P90"] = res_p90
        df.loc[cluster_overall_mask, "STAGE4_APPLIED_RESIDUAL_P95"] = res_p95
        df.loc[cluster_overall_mask, "STAGE4_THRESHOLD_SOURCE"] = thresh_source

        # Fast percentile ranking using searchsorted
        if len(surge_dist) > 0 and len(res_dist) > 0:
            df.loc[cluster_overall_mask, "STAGE4_RELATIVE_SURGE_PERCENTILE"] = (
                np.searchsorted(surge_dist, df.loc[cluster_overall_mask, "RELATIVE_SURGE_RATIO_FINAL"]) / len(surge_dist)
            )
            df.loc[cluster_overall_mask, "STAGE4_POSITIVE_RESIDUAL_PERCENTILE"] = (
                np.searchsorted(res_dist, df.loc[cluster_overall_mask, "STAGE4_POSITIVE_PAYMENT_RESIDUAL"]) / len(res_dist)
            )

    # ---------------------------------------------------------
    # 4. Risk Scoring and Tiers
    # ---------------------------------------------------------
    df.loc[eligible_mask, "STAGE4_PRICE_RISK_SCORE"] = (
        0.70 * df.loc[eligible_mask, "STAGE4_RELATIVE_SURGE_PERCENTILE"] +
        0.30 * df.loc[eligible_mask, "STAGE4_POSITIVE_RESIDUAL_PERCENTILE"]
    )

    df.loc[eligible_mask, "STAGE4_PRICE_TIER"] = "standard"

    elevated_mask = eligible_mask & (
        df["RELATIVE_SURGE_RATIO_FINAL"] >= df["STAGE4_APPLIED_SURGE_P95"]
    ) & (
        df["STAGE4_POSITIVE_PAYMENT_RESIDUAL"] >= df["STAGE4_APPLIED_RESIDUAL_P90"]
    )
    df.loc[elevated_mask, "STAGE4_PRICE_TIER"] = "elevated"
    df.loc[elevated_mask, "STAGE4_PRICE_FLAG"] = 1

    extreme_mask = eligible_mask & (
        df["RELATIVE_SURGE_RATIO_FINAL"] >= df["STAGE4_APPLIED_SURGE_P99"]
    ) & (
        df["STAGE4_POSITIVE_PAYMENT_RESIDUAL"] >= df["STAGE4_APPLIED_RESIDUAL_P95"]
    )
    df.loc[extreme_mask, "STAGE4_PRICE_TIER"] = "extreme"
    df.loc[extreme_mask, "STAGE4_PRICE_FLAG"] = 1

    # ---------------------------------------------------------
    # 5. Evaluation Correction
    # ---------------------------------------------------------
    if "SCENARIO_TYPE" in df.columns:
        df["IS_STAGE4_PRICE_SCENARIO"] = df["SCENARIO_TYPE"].isin([
            "extreme_payment_deviation",
            "moderate_payment_deviation"
        ]).astype(int)

    # ---------------------------------------------------------
    # 6. Assertions
    # ---------------------------------------------------------
    # 6a. Calibration purity assertions
    if "IS_ANOMALY_INJECTED" in clean_holdout_df.columns:
        assert clean_holdout_df["IS_ANOMALY_INJECTED"].sum() == 0, "Assertion Failed: Injected rows used in threshold calibration."
    if "SYNTHETIC_RECORD_CREATED" in clean_holdout_df.columns:
        assert clean_holdout_df["SYNTHETIC_RECORD_CREATED"].sum() == 0, "Assertion Failed: Synthetic duplicate rows used in threshold calibration."

    ineligible_mask = ~df["PRICE_SIGNAL_ELIGIBLE"]

    assert clean_holdout_df["IS_ANOMALY_INJECTED"].sum() == 0, "Assertion Failed: Injected rows used in threshold calibration."
    assert clean_holdout_df["SYNTHETIC_RECORD_CREATED"].sum() == 0, "Assertion Failed: Synthetic duplicate rows used in threshold calibration."

    assert df.loc[ineligible_mask, "STAGE4_PRICE_RISK_SCORE"].isna().all(), "Assertion Failed: Price-ineligible row has a non-null Stage 4 score."
    assert (df.loc[ineligible_mask, "STAGE4_PRICE_FLAG"] == 0).all(), "Assertion Failed: Price-ineligible row has STAGE4_PRICE_FLAG = 1."
    assert (df.loc[ineligible_mask, "STAGE4_PRICE_TIER"] == "not_eligible").all(), "Assertion Failed: Price-ineligible row has a tier other than 'not_eligible'."

    assert df.loc[eligible_mask, "STAGE4_PRICE_RISK_SCORE"].between(0, 1).all(), "Assertion Failed: Non-null Stage 4 scores are not between 0 and 1."

    required_threshold_cols = [
        "STAGE4_APPLIED_SURGE_P95",
        "STAGE4_APPLIED_SURGE_P99",
        "STAGE4_APPLIED_RESIDUAL_P90",
        "STAGE4_APPLIED_RESIDUAL_P95"
    ]
    assert df.loc[eligible_mask, required_threshold_cols].notna().all().all(), "Assertion Failed: Not all eligible rows have applied threshold values populated."

    assert df.loc[eligible_mask, "STAGE4_THRESHOLD_SOURCE"].isin(["cluster_holdout", "global_holdout_fallback"]).all(), "Assertion Failed: Invalid threshold source on eligible row."

    logging.info("Stage 4 Processing & Assertions completed successfully.")

    return df


def save_stage4_outputs(df: pd.DataFrame, output_dir: str = "data/processed"):
    """
    Saves Stage 4 scored output to parquet and csv, creating directories if missing.
    """
    os.makedirs(output_dir, exist_ok=True)

    parquet_path = os.path.abspath(os.path.join(output_dir, "cms_claims_stage4_price_scored.parquet"))
    csv_path = os.path.abspath(os.path.join(output_dir, "cms_claims_stage4_price_scored.csv"))

    df.to_parquet(parquet_path, index=False)
    df.to_csv(csv_path, index=False)

    print("\n" + "="*80)
    print("STAGE 4 FILE SAVE VERIFICATION")
    print("="*80)
    print(f"Parquet saved -> {parquet_path} ({os.path.getsize(parquet_path) / (1024*1024):.2f} MB)")
    print(f"CSV saved     -> {csv_path} ({os.path.getsize(csv_path) / (1024*1024):.2f} MB)")
    print("="*80)


def display_stage4_results(df: pd.DataFrame):
    """
    Prints a clean summary of Stage 4 execution directly in Colab output cell.
    """
    print("\n" + "="*80)
    print("STAGE 4 PRICE-DEVIATION ANOMALY ENGINE RESULTS SUMMARY")
    print("="*80)
    print(f"Total Claims Processed: {len(df):,}")
    print(f"Eligible Claims:       {df['PRICE_SIGNAL_ELIGIBLE'].sum():,} ({df['PRICE_SIGNAL_ELIGIBLE'].mean():.2%})")
    print(f"Ineligible Claims:     {(~df['PRICE_SIGNAL_ELIGIBLE']).sum():,}\n")

    print("--- TIER BREAKDOWN ---")
    tier_counts = df["STAGE4_PRICE_TIER"].value_counts(dropna=False)
    for tier, count in tier_counts.items():
        print(f"  {tier:<15}: {count:>8,} ({count/len(df):.2%})")

    print("\n--- PRICE FLAG BREAKDOWN ---")
    flag_counts = df["STAGE4_PRICE_FLAG"].value_counts(dropna=False)
    for flag, count in flag_counts.items():
        print(f"  Flag = {flag}: {count:>8,} ({count/len(df):.2%})")

    print("\n--- THRESHOLD SOURCE DISTRIBUTION ---")
    source_counts = df["STAGE4_THRESHOLD_SOURCE"].value_counts(dropna=False)
    for src, count in source_counts.items():
        print(f"  {src:<25}: {count:>8,} ({count/len(df):.2%})")

    eligible_scores = df.loc[df["PRICE_SIGNAL_ELIGIBLE"], "STAGE4_PRICE_RISK_SCORE"]
    if len(eligible_scores) > 0:
        print("\n--- RISK SCORE STATS (ELIGIBLE ROWS) ---")
        print(f"  Min:  {eligible_scores.min():.4f}")
        print(f"  P25:  {eligible_scores.quantile(0.25):.4f}")
        print(f"  Mean: {eligible_scores.mean():.4f}")
        print(f"  P75:  {eligible_scores.quantile(0.75):.4f}")
        print(f"  Max:  {eligible_scores.max():.4f}")

    print("\n--- SAMPLE OUTPUT PREVIEW ---")
    preview_cols = [col for col in [
        "STAGE4_CLUSTER_KEY", "PRICE_SIGNAL_ELIGIBLE",
        "STAGE4_PRICE_RISK_SCORE", "STAGE4_PRICE_TIER",
        "STAGE4_PRICE_FLAG", "STAGE4_THRESHOLD_SOURCE"
    ] if col in df.columns]

    # Display top flagged anomalies if present, else head
    sample_df = df[df["STAGE4_PRICE_FLAG"] == 1]
    if sample_df.empty:
        sample_df = df
    display_cols = preview_cols if len(preview_cols) > 0 else df.columns[:6]
    print(sample_df[display_cols].head(5).to_string(index=False))
    print("="*80 + "\n")


# ==============================================================================
# COLAB EXECUTION BLOCK
# Reads Stage 3 input, runs Stage 4 Engine, outputs summary, saves results
# ==============================================================================
input_path = "data/processed/cms_claims_stage3_calibrated.parquet"

if os.path.exists(input_path):
    print(f"Loading Stage 3 input from: {input_path}")
    df_stage3 = pd.read_parquet(input_path)

    # Run Stage 4 Pipeline
    df_stage4_scored = run_stage4_price_deviation_engine(df_stage3)

    # Save Outputs
    save_stage4_outputs(df_stage4_scored, output_dir="data/processed")

    # Print Results directly to Colab cell
    display_stage4_results(df_stage4_scored)
else:
    print(f"WARNING: Stage 3 file not found at '{input_path}'.")
    print("Please ensure the prior stage notebook has saved 'data/processed/cms_claims_stage3_calibrated.parquet'.")

Loading Stage 3 input from: data/processed/cms_claims_stage3_calibrated.parquet

STAGE 4 FILE SAVE VERIFICATION
Parquet saved -> /content/data/processed/cms_claims_stage4_price_scored.parquet (1.39 MB)
CSV saved     -> /content/data/processed/cms_claims_stage4_price_scored.csv (6.48 MB)

STAGE 4 PRICE-DEVIATION ANOMALY ENGINE RESULTS SUMMARY
Total Claims Processed: 10,105
Eligible Claims:       8,400 (83.13%)
Ineligible Claims:     1,705

--- TIER BREAKDOWN ---
  standard       :    7,914 (78.32%)
  not_eligible   :    1,705 (16.87%)
  elevated       :      379 (3.75%)
  extreme        :      107 (1.06%)

--- PRICE FLAG BREAKDOWN ---
  Flag = 0:    9,619 (95.19%)
  Flag = 1:      486 (4.81%)

--- THRESHOLD SOURCE DISTRIBUTION ---
  cluster_holdout          :    8,400 (83.13%)
  not_eligible             :    1,705 (16.87%)

--- RISK SCORE STATS (ELIGIBLE ROWS) ---
  Min:  0.0000
  P25:  0.1457
  Mean: 0.4368
  P75:  0.7492
  Max:  1.0000

--- SAMPLE OUTPUT PREVIEW ---
STAGE4_CLUSTER_KEY

In [11]:
import os
import logging
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


def evaluate_performance(df: pd.DataFrame):
    """
    Generates all requested evaluation tables and diagnostics with defensive safety guards.
    """
    logger.info("Evaluating top-of-queue and classification metrics...")
    df = df.copy()

    # ---------------------------------------------------------
    # 0. Defensive Column Normalization & Guarantees
    # ---------------------------------------------------------
    if "SCENARIO_TYPE" not in df.columns:
        df["SCENARIO_TYPE"] = "clean"

    if "IS_PRICE_DEVIATION_SCENARIO" not in df.columns:
        df["IS_PRICE_DEVIATION_SCENARIO"] = df["SCENARIO_TYPE"].isin([
            "extreme_payment_deviation", "moderate_payment_deviation"
        ]).astype(int)

    if "IS_ANOMALY_INJECTED" not in df.columns:
        df["IS_ANOMALY_INJECTED"] = (df["SCENARIO_TYPE"] != "clean").astype(int)

    if "PRICE_SIGNAL_ELIGIBLE" not in df.columns:
        df["PRICE_SIGNAL_ELIGIBLE"] = True

    if "STAGE4_PRICE_FLAG" not in df.columns:
        df["STAGE4_PRICE_FLAG"] = 0

    if "STAGE4_PRICE_RISK_SCORE" not in df.columns:
        df["STAGE4_PRICE_RISK_SCORE"] = 0.0

    if "STAGE4_PRICE_TIER" not in df.columns:
        df["STAGE4_PRICE_TIER"] = "standard"

    # Cluster column fallback
    if "kmeans_cluster" in df.columns:
        cluster_col = "kmeans_cluster"
    elif "STAGE4_CLUSTER_KEY" in df.columns:
        cluster_col = "STAGE4_CLUSTER_KEY"
    else:
        df["STAGE4_CLUSTER_KEY"] = "0"
        cluster_col = "STAGE4_CLUSTER_KEY"

    # ---------------------------------------------------------
    # 1. Primary Metrics
    # ---------------------------------------------------------
    primary_mask = (df["IS_ANOMALY_INJECTED"] == 0) | (df["IS_PRICE_DEVIATION_SCENARIO"] == 1)
    eval_df = df[primary_mask & (df["PRICE_SIGNAL_ELIGIBLE"] == True)].copy()

    y_true = eval_df["IS_PRICE_DEVIATION_SCENARIO"]
    y_pred = eval_df["STAGE4_PRICE_FLAG"]

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        clean_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    else:
        clean_fpr = 0.0

    tier_counts = eval_df["STAGE4_PRICE_TIER"].value_counts()

    ext_mask = eval_df["SCENARIO_TYPE"] == "extreme_payment_deviation"
    mod_mask = eval_df["SCENARIO_TYPE"] == "moderate_payment_deviation"

    ext_recall = eval_df[ext_mask]["STAGE4_PRICE_FLAG"].mean() if ext_mask.sum() > 0 else 0.0
    mod_recall = eval_df[mod_mask]["STAGE4_PRICE_FLAG"].mean() if mod_mask.sum() > 0 else 0.0

    primary_metrics = {
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Clean FPR": clean_fpr,
        "Elevated Count": tier_counts.get("elevated", 0),
        "Extreme Count": tier_counts.get("extreme", 0),
        "Extreme Scenario Recall": ext_recall,
        "Moderate Scenario Recall": mod_recall
    }

    # ---------------------------------------------------------
    # 2. Duplicate-like Billing Diagnostic
    # ---------------------------------------------------------
    dup_mask = (df["SCENARIO_TYPE"] == "duplicate_like_billing")
    dup_df = df[dup_mask]
    dup_diagnostic = {
        "Total Rows": len(dup_df),
        "Pct Eligible": dup_df["PRICE_SIGNAL_ELIGIBLE"].mean() if len(dup_df) > 0 else 0.0,
        "Pct Flagged": dup_df["STAGE4_PRICE_FLAG"].mean() if len(dup_df) > 0 else 0.0,
        "Median Score": dup_df["STAGE4_PRICE_RISK_SCORE"].median() if len(dup_df) > 0 else 0.0
    }

    # ---------------------------------------------------------
    # 3. Top of Queue Ranking Evaluation
    # ---------------------------------------------------------
    eval_df_sorted = eval_df.sort_values(by="STAGE4_PRICE_RISK_SCORE", ascending=False)
    total_eligible = len(eval_df_sorted)
    total_positives = y_true.sum()

    queue_tables = []
    for cutoff_pct in [0.05, 0.10]:
        k = int(total_eligible * cutoff_pct)
        top_k = eval_df_sorted.head(k)

        captured_pos = top_k["IS_PRICE_DEVIATION_SCENARIO"].sum()
        clean_included = (top_k["IS_ANOMALY_INJECTED"] == 0).sum()

        q_prec = captured_pos / k if k > 0 else 0.0
        q_rec = captured_pos / total_positives if total_positives > 0 else 0.0

        queue_tables.append({
            "Queue Threshold": f"Top {int(cutoff_pct*100)}%",
            "Number of Claims": k,
            "Price Scenarios Captured": captured_pos,
            "Clean Claims Included": clean_included,
            "Precision": f"{q_prec*100:.1f}%",
            "Recall": f"{q_rec*100:.1f}%"
        })

    queue_df = pd.DataFrame(queue_tables)

    # ---------------------------------------------------------
    # 4. Cluster-Level Diagnostics
    # ---------------------------------------------------------
    cluster_diags = []
    for c_id in df[cluster_col].dropna().unique():
        c_sub = eval_df[eval_df[cluster_col] == c_id]
        if len(c_sub) == 0:
            continue

        c_clean = c_sub[c_sub["IS_ANOMALY_INJECTED"] == 0]
        c_pos = c_sub[c_sub["IS_PRICE_DEVIATION_SCENARIO"] == 1]

        c_ext_mask = c_sub["SCENARIO_TYPE"] == "extreme_payment_deviation"
        c_mod_mask = c_sub["SCENARIO_TYPE"] == "moderate_payment_deviation"

        cfpr = c_clean["STAGE4_PRICE_FLAG"].mean() if len(c_clean) > 0 else 0.0
        thresh_src = c_sub["STAGE4_THRESHOLD_SOURCE"].iloc[0] if "STAGE4_THRESHOLD_SOURCE" in c_sub.columns and len(c_sub) > 0 else "N/A"

        cluster_diags.append({
            "Cluster": c_id,
            "Clean Eligible Claims": len(c_clean),
            "Price Deviation Scenarios": len(c_pos),
            "Total Flags": c_sub["STAGE4_PRICE_FLAG"].sum(),
            "Clean FPR": f"{cfpr*100:.2f}%",
            "Extreme Recall": f"{(c_sub[c_ext_mask]['STAGE4_PRICE_FLAG'].mean() if c_ext_mask.sum() > 0 else 0.0)*100:.1f}%",
            "Moderate Recall": f"{(c_sub[c_mod_mask]['STAGE4_PRICE_FLAG'].mean() if c_mod_mask.sum() > 0 else 0.0)*100:.1f}%",
            "Median Risk Score": f"{c_sub['STAGE4_PRICE_RISK_SCORE'].median():.3f}",
            "Threshold Source": thresh_src
        })

    cluster_df = pd.DataFrame(cluster_diags)

    return primary_metrics, dup_diagnostic, queue_df, cluster_df


def print_evaluation_report(primary_metrics, dup_diagnostic, queue_df, cluster_df):
    """
    Helper function to cleanly format and display the evaluation output.
    """
    print("\n" + "="*70)
    print("STAGE 4: PRIMARY EVALUATION METRICS")
    print("="*70)
    for k, v in primary_metrics.items():
        if isinstance(v, float):
            print(f"  {k:<28}: {v:.4f} ({v*100:.2f}%)" if "Count" not in k else f"  {k:<28}: {v:.4f}")
        else:
            print(f"  {k:<28}: {v:,}")

    print("\n" + "="*70)
    print("STAGE 4: DUPLICATE-LIKE BILLING DIAGNOSTIC")
    print("="*70)
    for k, v in dup_diagnostic.items():
        if isinstance(v, float):
            print(f"  {k:<28}: {v:.4f}" + (f" ({v*100:.2f}%)" if "Pct" in k else ""))
        else:
            print(f"  {k:<28}: {v:,}")

    print("\n" + "="*70)
    print("STAGE 4: TOP OF QUEUE RANKING EVALUATION")
    print("="*70)
    print(queue_df.to_string(index=False))

    print("\n" + "="*70)
    print("STAGE 4: CLUSTER-LEVEL DIAGNOSTICS")
    print("="*70)
    print(cluster_df.to_string(index=False))
    print("="*70 + "\n")


# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
file_path = "data/processed/cms_claims_stage4_price_scored.parquet"

if not os.path.exists(file_path):
    file_path = file_path.replace(".parquet", ".csv")

if os.path.exists(file_path):
    print(f"Loading scored Stage 4 data from: {file_path}...")
    df_scored = pd.read_parquet(file_path) if file_path.endswith(".parquet") else pd.read_csv(file_path)

    # Execute Evaluation
    primary_metrics, dup_diagnostic, queue_df, cluster_df = evaluate_performance(df_scored)

    # Print Clean Output
    print_evaluation_report(primary_metrics, dup_diagnostic, queue_df, cluster_df)
else:
    print(f"Error: Scored Stage 4 file not found at '{file_path}'. Please run the Stage 4 script first.")

Loading scored Stage 4 data from: data/processed/cms_claims_stage4_price_scored.parquet...

STAGE 4: PRIMARY EVALUATION METRICS
  Precision                   : 0.1971 (19.71%)
  Recall                      : 0.3862 (38.62%)
  F1 Score                    : 0.2610 (26.10%)
  Clean FPR                   : 0.0480 (4.80%)
  Elevated Count              : 376
  Extreme Count               : 106
  Extreme Scenario Recall     : 0.6013 (60.13%)
  Moderate Scenario Recall    : 0.0000 (0.00%)

STAGE 4: DUPLICATE-LIKE BILLING DIAGNOSTIC
  Total Rows                  : 105
  Pct Eligible                : 0.8381 (83.81%)
  Pct Flagged                 : 0.0381 (3.81%)
  Median Score                : 0.4353

STAGE 4: TOP OF QUEUE RANKING EVALUATION
Queue Threshold  Number of Claims  Price Scenarios Captured  Clean Claims Included Precision Recall
         Top 5%               415                        99                    316     23.9%  40.2%
        Top 10%               831                       12

# Stage 5: Cluster-Stratified Isolation Forest + Welch's t-Test

This stage serves as an Unsupervised Multi-Variable Anomaly Engine backed by Statistical Hypothesis Verification.

In [12]:
import os
import logging
import numpy as np
import pandas as pd
import joblib
from scipy.stats import percentileofscore, ttest_ind
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# =====================================================================
# 1. LOAD AND VALIDATE
# =====================================================================

def load_and_validate(input_path: str):
    logger.info(f"Loading Stage 4 data from {input_path}")
    df = pd.read_parquet(input_path)

    npi_candidates = ["PRF_PHYSN_NPI_1", "RNDR_NPI", "PRF_NPI", "ORG_NPI_NUM", "PROVIDER_NPI", "Provider_NPI", "NPI", "PRVDR_NUM", "PROVIDER_ID"]
    hcpcs_candidates = ["ORIGINAL_HCPCS_CD", "HCPCS_CD_1", "HCPCS_CD", "HCPCS", "CPT_CODE", "PROC_CODE"]
    working_pmt_candidates = ["WORKING_PMT_AMT", "LINE_NCH_PMT_AMT_1", "CLM_PMT_AMT", "PAYMENT_AMOUNT"]
    expected_pmt_candidates = ["EXPECTED_PMT_FINAL", "EXPECTED_PAYMENT", "EXPECTED_PMT", "PREDICTED_PAYMENT"]

    npi_col = next((c for c in npi_candidates if c in df.columns), None)
    hcpcs_col = next((c for c in hcpcs_candidates if c in df.columns), None)
    working_pmt_col = next((c for c in working_pmt_candidates if c in df.columns), None)
    expected_pmt_col = next((c for c in expected_pmt_candidates if c in df.columns), None)

    if not npi_col:
        raise ValueError(f"Could not find an NPI/Provider ID column in dataframe. Checked: {npi_candidates}")
    if not hcpcs_col:
        raise ValueError(f"Could not find a Procedure/HCPCS column in dataframe. Checked: {hcpcs_candidates}")
    if not working_pmt_col:
        raise ValueError(f"Could not find a Working Payment column in dataframe. Checked: {working_pmt_candidates}")
    if not expected_pmt_col:
        raise ValueError(f"Could not find an Expected Payment column in dataframe. Checked: {expected_pmt_candidates}")

    col_mapping = {
        "NPI": npi_col,
        "HCPCS": hcpcs_col,
        "WORKING_PMT": working_pmt_col,
        "EXPECTED_PMT": expected_pmt_col,
    }
    logger.info(f"Successfully mapped core columns: {col_mapping}")

    # --- ADDED: Flag Missing or Invalid HCPCS Rows ---
    df["STAGE5_MISSING_HCPCS_FLAG"] = (
        df[hcpcs_col].isna() |
        df[hcpcs_col].astype(str).str.strip().isin(["", "None", "nan", "NaN"])
    ).astype(int)

    if "kmeans_cluster" not in df.columns:
        raise ValueError("kmeans_cluster column is missing from input dataset.")

    df["STAGE5_CLUSTER_KEY"] = df["kmeans_cluster"].astype(str).str.strip()

    if "MODEL_PARTITION" not in df.columns:
        df["MODEL_PARTITION"] = "holdout"

    if "SOURCE_CLAIM_LINE_ID" in df.columns and "CLAIM_LINE_ID" in df.columns:
        partition_map = df.dropna(subset=['CLAIM_LINE_ID']).set_index('CLAIM_LINE_ID')['MODEL_PARTITION'].to_dict()
        missing_part = df["MODEL_PARTITION"].isna()
        df.loc[missing_part, "MODEL_PARTITION"] = df.loc[missing_part, "SOURCE_CLAIM_LINE_ID"].map(partition_map)
    df["MODEL_PARTITION"] = df["MODEL_PARTITION"].fillna("holdout")

    return df, col_mapping

# =====================================================================
# 2 & 3. PARTITIONS AND FEATURE ENGINEERING
# =====================================================================
def engineer_contextual_features(df: pd.DataFrame, col_map: dict) -> pd.DataFrame:
    logger.info("Defining partitions and engineering train-only contextual features...")

    npi_col = col_map["NPI"]
    hcpcs_col = col_map["HCPCS"]
    pmt_col = col_map["WORKING_PMT"]

    clean_original_mask = (df["SYNTHETIC_RECORD_CREATED"] == 0) & (df["IS_ANOMALY_INJECTED"] == 0)
    clean_train_mask = clean_original_mask & (df["MODEL_PARTITION"] == "train")

    train_df = df[clean_train_mask]

    # Provider-level aggregations (Train only)
    prov_stats = train_df.groupby(npi_col).agg(
        STAGE5_PROVIDER_CLAIM_VOLUME=(pmt_col, 'count'),
        STAGE5_PROVIDER_HCPCS_DIVERSITY=(hcpcs_col, 'nunique'),
        STAGE5_PROVIDER_MEDIAN_PAYMENT=(pmt_col, 'median'),
        STAGE5_PROVIDER_MEAN_PAYMENT=(pmt_col, 'mean')
    ).reset_index()

    # Provider-HCPCS level aggregations (Train only)
    prov_hcpcs_stats = train_df.groupby([npi_col, hcpcs_col]).agg(
        STAGE5_PROVIDER_HCPCS_COUNT=(pmt_col, 'count')
    ).reset_index()

    # Map back to full df
    df = df.merge(prov_stats, on=npi_col, how='left')
    df = df.merge(prov_hcpcs_stats, on=[npi_col, hcpcs_col], how='left')

    # Unseen provider logic
    df["STAGE5_UNSEEN_PROVIDER_FLAG"] = df["STAGE5_PROVIDER_CLAIM_VOLUME"].isna().astype(int)

    # Impute unseen
    df["STAGE5_PROVIDER_CLAIM_VOLUME"] = df["STAGE5_PROVIDER_CLAIM_VOLUME"].fillna(0)
    df["STAGE5_PROVIDER_HCPCS_COUNT"] = df["STAGE5_PROVIDER_HCPCS_COUNT"].fillna(0)

    # Concentration
    df["STAGE5_PROVIDER_HCPCS_CONCENTRATION"] = np.where(
        df["STAGE5_PROVIDER_CLAIM_VOLUME"] > 0,
        df["STAGE5_PROVIDER_HCPCS_COUNT"] / df["STAGE5_PROVIDER_CLAIM_VOLUME"],
        0.0
    )

    global_median_pmt = train_df[pmt_col].median()
    global_mean_pmt = train_df[pmt_col].mean()

    df["STAGE5_PROVIDER_MEDIAN_PAYMENT"] = df["STAGE5_PROVIDER_MEDIAN_PAYMENT"].fillna(global_median_pmt)
    df["STAGE5_PROVIDER_MEAN_PAYMENT"] = df["STAGE5_PROVIDER_MEAN_PAYMENT"].fillna(global_mean_pmt)
    df["STAGE5_PROVIDER_HCPCS_DIVERSITY"] = df["STAGE5_PROVIDER_HCPCS_DIVERSITY"].fillna(1)

    # Log Transforms
    df["LOG_WORKING_PAYMENT"] = np.log1p(df[pmt_col].clip(lower=0))
    df["LOG_EXPECTED_PAYMENT"] = np.log1p(df[col_map["EXPECTED_PMT"]].clip(lower=0))
    df["LOG_PROVIDER_CLAIM_VOLUME"] = np.log1p(df["STAGE5_PROVIDER_CLAIM_VOLUME"].clip(lower=0))
    df["LOG_PROVIDER_HCPCS_COUNT"] = np.log1p(df["STAGE5_PROVIDER_HCPCS_COUNT"].clip(lower=0))

    if "PEER_GROUP_COUNT" in df.columns:
        df["LOG_PEER_GROUP_COUNT"] = np.log1p(df["PEER_GROUP_COUNT"].clip(lower=0))
    else:
        df["LOG_PEER_GROUP_COUNT"] = 0.0

    return df

# =====================================================================
# 4, 5, 6, 7. FEATURE SELECTION, TRAINING, AND SCORING
# =====================================================================

def build_and_score_models(df: pd.DataFrame, model_out_dir: str):
    logger.info("Building cluster-stratified Isolation Forest models...")

    candidate_features = [
        "RELATIVE_SURGE_RATIO_FINAL", "STAGE4_POSITIVE_PAYMENT_RESIDUAL",
        "STAGE4_RELATIVE_SURGE_PERCENTILE", "STAGE4_POSITIVE_RESIDUAL_PERCENTILE",
        "LOG_WORKING_PAYMENT", "LOG_EXPECTED_PAYMENT", "LOG_PEER_GROUP_COUNT",
        "LOG_PROVIDER_CLAIM_VOLUME", "LOG_PROVIDER_HCPCS_COUNT",
        "STAGE5_PROVIDER_HCPCS_CONCENTRATION", "CHRONIC_CONDITION_COUNT",
        "PATIENT_AGE", "AGE_MISSING_FLAG"
    ]

    features = [f for f in candidate_features if f in df.columns]
    logger.info(f"Selected candidate features ({len(features)}): {features}")

    df["STAGE5_MODEL_SOURCE"] = "unassigned"
    df["STAGE5_IF_RAW_ANOMALY_SCORE"] = np.nan
    df["STAGE5_IF_RISK_SCORE"] = np.nan
    df["STAGE5_SCORE_THRESHOLD_SOURCE"] = "unassigned"

    # --- ADDED: Define Eligibility Mask & Update Clean Training Masks ---
    stage5_score_eligible_mask = (df["STAGE5_MISSING_HCPCS_FLAG"] == 0)
    missing_hcpcs_mask = (df["STAGE5_MISSING_HCPCS_FLAG"] == 1)

    clean_original_mask = (
        (df["SYNTHETIC_RECORD_CREATED"] == 0) &
        (df["IS_ANOMALY_INJECTED"] == 0) &
        stage5_score_eligible_mask
    )
    clean_train_mask = clean_original_mask & (df["MODEL_PARTITION"] == "train")
    clean_holdout_mask = clean_original_mask & (df["MODEL_PARTITION"] == "holdout")

    clusters = df["STAGE5_CLUSTER_KEY"].unique()
    os.makedirs(model_out_dir, exist_ok=True)

    # Global Fallback Training
    global_train_df = df[clean_train_mask]
    if len(global_train_df) == 0:
        logger.warning("clean_train_mask returned 0 records! Falling back to clean_original_mask for global fit.")
        global_train_df = df[clean_original_mask]

    global_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median")),
        ('scaler', RobustScaler()),
        ('iforest', IsolationForest(n_estimators=300, max_samples="auto", contamination="auto", random_state=42, n_jobs=-1))
    ])

    global_pipeline.fit(global_train_df[features])

    holdout_subset = df[clean_holdout_mask]
    if len(holdout_subset) == 0:
        holdout_subset = global_train_df

    global_holdout_scores = -global_pipeline.score_samples(holdout_subset[features])

    for cluster in clusters:
        # Score ONLY eligible rows within this cluster
        cluster_mask = (df["STAGE5_CLUSTER_KEY"] == cluster) & stage5_score_eligible_mask
        c_train_mask = clean_train_mask & (df["STAGE5_CLUSTER_KEY"] == cluster)
        c_train_df = df[c_train_mask]

        if len(c_train_df) > 0:
            c_features = [f for f in features if c_train_df[f].nunique() > 1]
        else:
            c_features = []

        if len(c_train_df) >= 200 and len(c_features) > 0:
            logger.info(f"Training cluster '{cluster}' model on {len(c_train_df)} clean records. Features: {len(c_features)}")
            pipeline = Pipeline([
                ('imputer', SimpleImputer(strategy="median")),
                ('scaler', RobustScaler()),
                ('iforest', IsolationForest(n_estimators=300, max_samples="auto", contamination="auto", random_state=42, n_jobs=-1))
            ])
            pipeline.fit(c_train_df[c_features])
            joblib.dump(pipeline, os.path.join(model_out_dir, f"stage5_isolation_forest_cluster_{cluster}.joblib"))

            raw_scores = -pipeline.score_samples(df.loc[cluster_mask, c_features])
            df.loc[cluster_mask, "STAGE5_IF_RAW_ANOMALY_SCORE"] = raw_scores
            df.loc[cluster_mask, "STAGE5_MODEL_SOURCE"] = "cluster_specific"

            c_holdout_mask = clean_holdout_mask & (df["STAGE5_CLUSTER_KEY"] == cluster)
            if c_holdout_mask.sum() > 50:
                calib_scores = -pipeline.score_samples(df.loc[c_holdout_mask, c_features])
                df.loc[cluster_mask, "STAGE5_IF_RISK_SCORE"] = df.loc[cluster_mask, "STAGE5_IF_RAW_ANOMALY_SCORE"].apply(
                    lambda x: percentileofscore(calib_scores, x) / 100.0
                )
                df.loc[cluster_mask, "STAGE5_SCORE_THRESHOLD_SOURCE"] = f"cluster_{cluster}_holdout"
            else:
                df.loc[cluster_mask, "STAGE5_IF_RISK_SCORE"] = df.loc[cluster_mask, "STAGE5_IF_RAW_ANOMALY_SCORE"].apply(
                    lambda x: percentileofscore(global_holdout_scores, x) / 100.0
                )
                df.loc[cluster_mask, "STAGE5_SCORE_THRESHOLD_SOURCE"] = "global_holdout_fallback"
        else:
            logger.info(f"Cluster '{cluster}' insufficient for local model ({len(c_train_df)} rows). Using global fallback.")
            df.loc[cluster_mask, "STAGE5_IF_RAW_ANOMALY_SCORE"] = -global_pipeline.score_samples(df.loc[cluster_mask, features])
            df.loc[cluster_mask, "STAGE5_MODEL_SOURCE"] = "global_clean_fallback"
            df.loc[cluster_mask, "STAGE5_IF_RISK_SCORE"] = df.loc[cluster_mask, "STAGE5_IF_RAW_ANOMALY_SCORE"].apply(
                lambda x: percentileofscore(global_holdout_scores, x) / 100.0
            )
            df.loc[cluster_mask, "STAGE5_SCORE_THRESHOLD_SOURCE"] = "global_holdout_fallback"

    # Assign Tiers & Flags for Eligible Scored Rows
    df["STAGE5_IF_TIER"] = "standard"
    df.loc[stage5_score_eligible_mask & (df["STAGE5_IF_RISK_SCORE"] >= 0.95), "STAGE5_IF_TIER"] = "elevated"
    df.loc[stage5_score_eligible_mask & (df["STAGE5_IF_RISK_SCORE"] >= 0.99), "STAGE5_IF_TIER"] = "extreme"

    df["STAGE5_IF_FLAG"] = df["STAGE5_IF_TIER"].isin(["elevated", "extreme"]).astype(int)

    # --- ADDED: Explicit Handling for Missing-HCPCS Rows ---
    if missing_hcpcs_mask.any():
        logger.info(f"Flagged {missing_hcpcs_mask.sum()} records with missing HCPCS codes for data quality review.")
        df.loc[missing_hcpcs_mask, "STAGE5_IF_RAW_ANOMALY_SCORE"] = np.nan
        df.loc[missing_hcpcs_mask, "STAGE5_IF_RISK_SCORE"] = np.nan
        df.loc[missing_hcpcs_mask, "STAGE5_IF_TIER"] = "data_quality_review"
        df.loc[missing_hcpcs_mask, "STAGE5_IF_FLAG"] = 0
        df.loc[missing_hcpcs_mask, "STAGE5_MODEL_SOURCE"] = "not_scored_missing_hcpcs"
        df.loc[missing_hcpcs_mask, "STAGE5_SCORE_THRESHOLD_SOURCE"] = "not_applicable"

    return df, features

# =====================================================================
# 8 & 9. EVALUATION AND WELCH INFERENCE CHECK
# =====================================================================
def evaluate_and_test(df: pd.DataFrame):
    logger.info("Executing Stage 5 Evaluation and Welch's t-Test...")

    df["IS_STAGE5_EVALUATION_SCENARIO"] = df["SCENARIO_TYPE"].isin([
        "extreme_payment_deviation", "moderate_payment_deviation", "duplicate_like_billing"
    ]).astype(int)

    clean_holdout = df[(df["SYNTHETIC_RECORD_CREATED"] == 0) &
                       (df["IS_ANOMALY_INJECTED"] == 0) &
                       (df["MODEL_PARTITION"] == "holdout")]

    logger.info("=== Queue Metrics (Holdout + Injected) ===")
    eval_pool = pd.concat([clean_holdout, df[df["IS_STAGE5_EVALUATION_SCENARIO"] == 1]])

    def calc_queue(mask_name, mask):
        sub_df = eval_pool[eval_pool["IS_ANOMALY_INJECTED"] == 0 | mask]
        sub_df = sub_df.sort_values(by="STAGE5_IF_RISK_SCORE", ascending=False)
        positives = mask.sum()
        for pct in [0.05, 0.10]:
            k = int(len(sub_df) * pct)
            top_k = sub_df.head(k)
            captured = top_k["IS_STAGE5_EVALUATION_SCENARIO"].sum()
            logger.info(f"{mask_name} @ {pct*100:.0f}%: Precision={captured/k if k else 0:.2%} | Recall={captured/positives if positives else 0:.2%}")

    ext_mask = eval_pool["SCENARIO_TYPE"] == "extreme_payment_deviation"
    mod_mask = eval_pool["SCENARIO_TYPE"] == "moderate_payment_deviation"
    dup_mask = eval_pool["SCENARIO_TYPE"] == "duplicate_like_billing"
    all_mask = eval_pool["IS_STAGE5_EVALUATION_SCENARIO"] == 1

    calc_queue("All Scenarios", all_mask)
    calc_queue("Extreme", ext_mask)
    calc_queue("Moderate", mod_mask)
    calc_queue("Duplicates", dup_mask)

    # Welch's T-Test
    logger.info("=== Optional Welch's t-Test Inference Check ===")
    logger.info("DISCLAIMER: Welch’s t-test is an exploratory comparison of score distributions "
                "in this synthetic POC. It does not establish clinical validity, payment error, "
                "fraud, causality, or production performance.")

    clean_scores = clean_holdout["STAGE5_IF_RISK_SCORE"].dropna()

    scenarios = {
        "Extreme Payment": df[df["SCENARIO_TYPE"] == "extreme_payment_deviation"],
        "Moderate Payment": df[df["SCENARIO_TYPE"] == "moderate_payment_deviation"],
        "Duplicate Billing": df[df["SCENARIO_TYPE"] == "duplicate_like_billing"],
        "All Injected": df[df["IS_STAGE5_EVALUATION_SCENARIO"] == 1]
    }

    for name, sub_df in scenarios.items():
        scen_scores = sub_df["STAGE5_IF_RISK_SCORE"].dropna()
        if len(scen_scores) == 0:
            continue

        t_stat, p_val = ttest_ind(clean_scores, scen_scores, equal_var=False, nan_policy="omit")

        # Cohen's d
        mean_c, mean_s = clean_scores.mean(), scen_scores.mean()
        var_c, var_s = clean_scores.var(), scen_scores.var()
        pooled_std = np.sqrt((var_c + var_s) / 2)
        cohens_d = (mean_s - mean_c) / pooled_std if pooled_std > 0 else 0

        logger.info(f"Comparison: Clean Holdout vs {name}")
        logger.info(f"  Clean N: {len(clean_scores)} | Scenario N: {len(scen_scores)}")
        logger.info(f"  Clean Mean: {mean_c:.4f} | Scenario Mean: {mean_s:.4f} | Diff: {mean_s - mean_c:.4f}")
        logger.info(f"  t-stat: {t_stat:.4f} | p-value: {p_val:.4e} | Cohen's d: {cohens_d:.4f}\n")

# =====================================================================
# 10 & 11. ASSERTIONS AND OUTPUT
# =====================================================================
def run_assertions(df: pd.DataFrame, features: list):
    logger.info("Running quality assertions...")

    # Model leak checks
    leak_cols = ["IS_ANOMALY_INJECTED", "SCENARIO_TYPE", "STAGE4_PRICE_FLAG", "NPI", "CLAIM_ID"]
    for col in leak_cols:
        assert not any(col in f for f in features), f"Assertion Failed: {col} leaked into features."

    assert df["STAGE5_IF_RISK_SCORE"].between(0.0, 1.0).all() or df["STAGE5_IF_RISK_SCORE"].isna().all() == False, "Risk scores bounds invalid"
    assert not df["STAGE5_MODEL_SOURCE"].isnull().any(), "Missing model source assignments"

    # Threshold exceedance approx (clean holdout)
    c_holdout = df[(df["SYNTHETIC_RECORD_CREATED"] == 0) & (df["IS_ANOMALY_INJECTED"] == 0) & (df["MODEL_PARTITION"] == "holdout")]
    if len(c_holdout) > 0:
        p95_rate = (c_holdout["STAGE5_IF_RISK_SCORE"] >= 0.95).mean()
        p99_rate = (c_holdout["STAGE5_IF_RISK_SCORE"] >= 0.99).mean()
        logger.info(f"Clean Holdout Exceedance -> P95 Rate: {p95_rate:.2%}, P99 Rate: {p99_rate:.2%}")

    assert df["STAGE5_CLUSTER_KEY"].dtype == 'object', "Cluster key must be string"
    assert not np.isinf(df[features].values).any(), "Infinite values found in features"

def main():
    input_path = "data/processed/cms_claims_stage4_price_scored.parquet"
    out_parquet = "data/processed/cms_claims_stage5_iforest_scored.parquet"
    out_csv = "data/processed/cms_claims_stage5_iforest_scored.csv"
    model_dir = "models/"

    # Execution Flow
    df, col_map = load_and_validate(input_path)
    df = engineer_contextual_features(df, col_map)
    df, features = build_and_score_models(df, model_dir)

    run_assertions(df, features)
    evaluate_and_test(df)

    # Filter final columns to save
    output_cols = [c for c in df.columns if c not in ['STAGE4_POSITIVE_PAYMENT_RESIDUAL']] # Keep all core + requested

    logger.info(f"Saving Stage 5 outputs to {out_parquet}...")
    os.makedirs(os.path.dirname(out_parquet), exist_ok=True)
    df.to_parquet(out_parquet, index=False)
    df.to_csv(out_csv, index=False)
    logger.info("Stage 5 Execution Complete.")

if __name__ == "__main__":
    main()

In [13]:
import os
import pandas as pd
import numpy as np

# -------------------------------------------------------------------------
# 1. Load Stage 5 Output Dataset
# -------------------------------------------------------------------------
parquet_path = "data/processed/cms_claims_stage5_iforest_scored.parquet"
csv_path = "data/processed/cms_claims_stage5_iforest_scored.csv"

if os.path.exists(parquet_path):
    print(f"Loading from: {parquet_path}")
    df = pd.read_parquet(parquet_path)
elif os.path.exists(csv_path):
    print(f"Loading from: {csv_path}")
    df = pd.read_csv(csv_path)
else:
    raise FileNotFoundError("Stage 5 output file not found. Ensure Stage 5 script has completed.")

print(f"Dataset successfully loaded. Total rows: {len(df):,}\n")

# -------------------------------------------------------------------------
# 2. Stage 5 Tier & Flag Summary
# -------------------------------------------------------------------------
print("=" * 75)
print("1. STAGE 5 ANOMALY RISK TIERS & FLAGS")
print("=" * 75)
tier_summary = df.groupby(["STAGE5_IF_TIER", "STAGE5_IF_FLAG"]).size().reset_index(name="Claim_Count")
tier_summary["Percentage"] = (tier_summary["Claim_Count"] / len(df) * 100).map("{:.2f}%".format)
print(tier_summary.to_string(index=False))

# -------------------------------------------------------------------------
# 3. Performance Across Injected Scenarios
# -------------------------------------------------------------------------
print("\n" + "=" * 75)
print("2. RISK SCORES BY SCENARIO TYPE")
print("=" * 75)
if "SCENARIO_TYPE" in df.columns:
    scen_summary = df.groupby("SCENARIO_TYPE").agg(
        Total_Claims=("STAGE5_IF_RISK_SCORE", "count"),
        Mean_Risk_Score=("STAGE5_IF_RISK_SCORE", "mean"),
        Median_Risk_Score=("STAGE5_IF_RISK_SCORE", "median"),
        Flagged_Count=("STAGE5_IF_FLAG", "sum"),
        Pct_Flagged=("STAGE5_IF_FLAG", "mean")
    ).reset_index()

    scen_summary["Pct_Flagged"] = (scen_summary["Pct_Flagged"] * 100).map("{:.2f}%".format)
    scen_summary["Mean_Risk_Score"] = scen_summary["Mean_Risk_Score"].map("{:.4f}".format)
    scen_summary["Median_Risk_Score"] = scen_summary["Median_Risk_Score"].map("{:.4f}".format)
    print(scen_summary.to_string(index=False))

# -------------------------------------------------------------------------
# 4. Cluster-Level Diagnostics
# -------------------------------------------------------------------------
print("\n" + "=" * 75)
print("3. CLUSTER-LEVEL BREAKDOWN")
print("=" * 75)
if "STAGE5_CLUSTER_KEY" in df.columns:
    cluster_summary = df.groupby("STAGE5_CLUSTER_KEY").agg(
        Total_Claims=("STAGE5_IF_RISK_SCORE", "count"),
        Model_Source=("STAGE5_MODEL_SOURCE", lambda x: x.iloc[0] if len(x) > 0 else "N/A"),
        Median_Risk_Score=("STAGE5_IF_RISK_SCORE", "median"),
        Elevated_Count=("STAGE5_IF_TIER", lambda x: (x == "elevated").sum()),
        Extreme_Count=("STAGE5_IF_TIER", lambda x: (x == "extreme").sum())
    ).reset_index()

    cluster_summary["Median_Risk_Score"] = cluster_summary["Median_Risk_Score"].map("{:.4f}".format)
    print(cluster_summary.to_string(index=False))

# -------------------------------------------------------------------------
# 5. Top 5 High-Risk Queue Preview
# -------------------------------------------------------------------------
print("\n" + "=" * 75)
print("4. TOP 5 HIGHEST RISK CLAIMS (QUEUE PREVIEW)")
print("=" * 75)
display_cols = [
    col for col in [
        "CLAIM_LINE_ID", "DESYNPUF_ID", "PRF_PHYSN_NPI_1", "ORIGINAL_HCPCS_CD", "HCPCS_CD_1",
        "SCENARIO_TYPE", "STAGE5_CLUSTER_KEY", "STAGE5_IF_RAW_ANOMALY_SCORE",
        "STAGE5_IF_RISK_SCORE", "STAGE5_IF_TIER"
    ] if col in df.columns
]

top_10 = df.sort_values(by="STAGE5_IF_RISK_SCORE", ascending=False).head(5)
print(top_10[display_cols].to_string(index=False))
print("=" * 75 + "\n")

Loading from: data/processed/cms_claims_stage5_iforest_scored.parquet
Dataset successfully loaded. Total rows: 10,105

1. STAGE 5 ANOMALY RISK TIERS & FLAGS
     STAGE5_IF_TIER  STAGE5_IF_FLAG  Claim_Count Percentage
data_quality_review               0           91      0.90%
           elevated               1          224      2.22%
            extreme               1           96      0.95%
           standard               0         9694     95.93%

2. RISK SCORES BY SCENARIO TYPE
             SCENARIO_TYPE  Total_Claims Mean_Risk_Score Median_Risk_Score  Flagged_Count Pct_Flagged
                     clean          9663          0.2286            0.0025            211       2.16%
    duplicate_like_billing           105          0.2431            0.0025              6       5.71%
 extreme_payment_deviation           158          0.9244            0.9795             94      59.49%
moderate_payment_deviation            88          0.6712            0.6996              9      10.23%


In [14]:
print(df.groupby(["STAGE5_CLUSTER_KEY", "MODEL_PARTITION"]).size())

STAGE5_CLUSTER_KEY  MODEL_PARTITION
0                   holdout            1240
                    train              4770
                    unassigned          139
1                   holdout             410
                    train              1688
                    unassigned           62
2                   holdout             318
                    train              1433
                    unassigned           45
dtype: int64


#Variational Autoencoders (VAE)

#PyTorch Graph Neural Network (GNN) Implementation

#Stage 6: PARALLEL RISK-FUSION / META-ENSEMBLE COMBINER

In [15]:
"""
STAGE 6: TRANSPARENT PARALLEL RISK-FUSION / META-ENSEMBLE COMBINER
------------------------------------------------------------------
Combines Stage 4 (Price-Deviation) and Stage 5 (Isolation Forest)
continuous risk scores into a single, explainable Audit Priority Score
for human payment-integrity review.
"""

import os
import logging
import numpy as np
import pandas as pd

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Config
INPUT_FILE = "data/processed/cms_claims_stage5_iforest_scored.parquet"
OUT_DIR = "data/processed"
OUT_PARQUET = os.path.join(OUT_DIR, "cms_claims_stage6_ensemble_scored.parquet")
OUT_CSV = os.path.join(OUT_DIR, "cms_claims_stage6_ensemble_scored.csv")

# Fusion Weights
W_STAGE4 = 0.55
W_STAGE5 = 0.45

def load_and_validate(input_path: str) -> pd.DataFrame:
    """Loads dataset and validates presence and bounds of required scores."""
    logger.info(f"Loading Stage 5 output from {input_path}")
    df = pd.read_parquet(input_path)

    score_cols = ["STAGE4_PRICE_RISK_SCORE", "STAGE5_IF_RISK_SCORE"]
    available_scores = [c for c in score_cols if c in df.columns]

    if not available_scores:
        raise ValueError(f"Neither Stage 4 nor Stage 5 risk score columns found. Looked for {score_cols}")

    # Validate bounds (0 to 1) for existing scores
    for col in available_scores:
        out_of_bounds = df.loc[df[col].notna(), col].apply(lambda x: x < 0.0 or x > 1.0)
        if out_of_bounds.any():
            raise ValueError(f"Scores in {col} are outside the required [0, 1] range.")

    # Ensure Stage4/Stage5 missing scores are explicitly NaN, not just missing columns
    for col in score_cols:
        if col not in df.columns:
            df[col] = np.nan

    return df

def define_valid_signals(df: pd.DataFrame) -> pd.DataFrame:
    """Creates boolean availability and data-quality flags."""
    df["STAGE6_STAGE4_AVAILABLE"] = df["STAGE4_PRICE_RISK_SCORE"].notna()
    df["STAGE6_STAGE5_AVAILABLE"] = df["STAGE5_IF_RISK_SCORE"].notna()

    df["STAGE6_DATA_QUALITY_REVIEW"] = False

    # Check upstream data quality flags
    if "STAGE5_MISSING_HCPCS_FLAG" in df.columns:
        df["STAGE6_DATA_QUALITY_REVIEW"] = df["STAGE6_DATA_QUALITY_REVIEW"] | (df["STAGE5_MISSING_HCPCS_FLAG"] == 1)

    if "STAGE5_DATA_QUALITY_FLAG" in df.columns:
        df["STAGE6_DATA_QUALITY_REVIEW"] = df["STAGE6_DATA_QUALITY_REVIEW"] | (df["STAGE5_DATA_QUALITY_FLAG"] == 1)

    # Also flag if both scores are entirely missing
    missing_both = (~df["STAGE6_STAGE4_AVAILABLE"]) & (~df["STAGE6_STAGE5_AVAILABLE"])
    df.loc[missing_both, "STAGE6_DATA_QUALITY_REVIEW"] = True

    return df

def fuse_scores(df: pd.DataFrame) -> pd.DataFrame:
    """Combines Stage 4 and Stage 5 scores dynamically based on availability."""
    df["STAGE6_ENSEMBLE_RISK_SCORE"] = np.nan
    df["STAGE6_FUSION_SOURCE"] = "not_scored"

    # Masks
    dq_issue = df["STAGE6_DATA_QUALITY_REVIEW"]
    has_both = df["STAGE6_STAGE4_AVAILABLE"] & df["STAGE6_STAGE5_AVAILABLE"] & ~dq_issue
    has_s4_only = df["STAGE6_STAGE4_AVAILABLE"] & ~df["STAGE6_STAGE5_AVAILABLE"] & ~dq_issue
    has_s5_only = ~df["STAGE6_STAGE4_AVAILABLE"] & df["STAGE6_STAGE5_AVAILABLE"] & ~dq_issue

    # Apply logic
    df.loc[has_both, "STAGE6_ENSEMBLE_RISK_SCORE"] = (W_STAGE4 * df.loc[has_both, "STAGE4_PRICE_RISK_SCORE"]) + \
                                                     (W_STAGE5 * df.loc[has_both, "STAGE5_IF_RISK_SCORE"])
    df.loc[has_both, "STAGE6_FUSION_SOURCE"] = "stage4_and_stage5"

    df.loc[has_s4_only, "STAGE6_ENSEMBLE_RISK_SCORE"] = df.loc[has_s4_only, "STAGE4_PRICE_RISK_SCORE"]
    df.loc[has_s4_only, "STAGE6_FUSION_SOURCE"] = "stage4_only"

    df.loc[has_s5_only, "STAGE6_ENSEMBLE_RISK_SCORE"] = df.loc[has_s5_only, "STAGE5_IF_RISK_SCORE"]
    df.loc[has_s5_only, "STAGE6_FUSION_SOURCE"] = "stage5_only"

    df["AUDIT_PRIORITY_SCORE"] = df["STAGE6_ENSEMBLE_RISK_SCORE"] * 100.0

    return df

def calibrate_final_tiers(df: pd.DataFrame) -> pd.DataFrame:
    """Determines operation thresholds using strictly clean holdout records."""
    clean_holdout_ensemble_mask = (
        (df["SYNTHETIC_RECORD_CREATED"] == 0) &
        (df["IS_ANOMALY_INJECTED"] == 0) &
        (df["MODEL_PARTITION"] == "holdout") &
        df["STAGE6_ENSEMBLE_RISK_SCORE"].notna() &
        (~df["STAGE6_DATA_QUALITY_REVIEW"])
    )

    clean_holdout = df[clean_holdout_ensemble_mask]
    if len(clean_holdout) == 0:
        logger.warning("No clean holdout records available! Using all available scored clean records for calibration.")
        clean_fallback_mask = (df["SYNTHETIC_RECORD_CREATED"] == 0) & (df["IS_ANOMALY_INJECTED"] == 0) & df["STAGE6_ENSEMBLE_RISK_SCORE"].notna()
        clean_holdout = df[clean_fallback_mask]

    p95 = clean_holdout["STAGE6_ENSEMBLE_RISK_SCORE"].quantile(0.95)
    p99 = clean_holdout["STAGE6_ENSEMBLE_RISK_SCORE"].quantile(0.99)

    logger.info(f"Clean Holdout Calibration (N={len(clean_holdout)}): P95={p95:.4f}, P99={p99:.4f}")

    df["STAGE6_THRESHOLD_SOURCE"] = "not_scored"
    df["STAGE6_AUDIT_TIER"] = "not_scored"
    df["STAGE6_AUDIT_FLAG"] = 0

    scored = df["STAGE6_ENSEMBLE_RISK_SCORE"].notna()

    # Assign threshold sources
    df.loc[scored, "STAGE6_THRESHOLD_SOURCE"] = "clean_holdout_percentile"

    # Assign tiers
    is_standard = scored & (df["STAGE6_ENSEMBLE_RISK_SCORE"] < p95)
    is_elevated = scored & (df["STAGE6_ENSEMBLE_RISK_SCORE"] >= p95) & (df["STAGE6_ENSEMBLE_RISK_SCORE"] < p99)
    is_extreme = scored & (df["STAGE6_ENSEMBLE_RISK_SCORE"] >= p99)

    df.loc[is_standard, "STAGE6_AUDIT_TIER"] = "standard"
    df.loc[is_elevated, "STAGE6_AUDIT_TIER"] = "elevated"
    df.loc[is_extreme, "STAGE6_AUDIT_TIER"] = "extreme"

    df.loc[is_elevated | is_extreme, "STAGE6_AUDIT_FLAG"] = 1

    return df

def run_quality_assertions(df: pd.DataFrame):
    """Enforces strict constraints on scoring and data usage."""
    # Score bounds
    scored = df["STAGE6_ENSEMBLE_RISK_SCORE"].notna()
    assert df.loc[scored, "STAGE6_ENSEMBLE_RISK_SCORE"].between(0, 1).all(), "Ensemble score outside [0,1]"
    assert df.loc[scored, "AUDIT_PRIORITY_SCORE"].between(0, 100).all(), "Audit priority score outside [0,100]"

    # DQ row checks
    dq = df["STAGE6_DATA_QUALITY_REVIEW"]
    assert df.loc[dq, "STAGE6_ENSEMBLE_RISK_SCORE"].isna().all(), "DQ rows must have null ensemble score"
    assert (df.loc[dq, "STAGE6_AUDIT_TIER"] == "not_scored").all(), "DQ rows must be not_scored tier"
    assert (df.loc[dq, "STAGE6_AUDIT_FLAG"] == 0).all(), "DQ rows cannot be flagged"
    assert (df.loc[scored, "STAGE6_THRESHOLD_SOURCE"] == "clean_holdout_percentile").all(), "Scored rows lack holdout source"

    # Calibration leak check
    clean_holdout_ensemble_mask = (
        (df["SYNTHETIC_RECORD_CREATED"] == 0) &
        (df["IS_ANOMALY_INJECTED"] == 0) &
        (df["MODEL_PARTITION"] == "holdout") &
        df["STAGE6_ENSEMBLE_RISK_SCORE"].notna() &
        (~df["STAGE6_DATA_QUALITY_REVIEW"])
    )
    calib_df = df[clean_holdout_ensemble_mask]
    assert (calib_df["IS_ANOMALY_INJECTED"] == 0).all(), "Clean-holdout contains injected scenarios"
    assert (calib_df["SYNTHETIC_RECORD_CREATED"] == 0).all(), "Clean-holdout contains synthetic duplicate rows"

def evaluate_performance(df: pd.DataFrame):
    """Calculates and prints operational metrics for evaluation only."""
    df["IS_STAGE6_PRICE_SCENARIO"] = df["SCENARIO_TYPE"].isin(["extreme_payment_deviation", "moderate_payment_deviation"]).astype(int)
    df["IS_STAGE6_INJECTED_SCENARIO"] = df["SCENARIO_TYPE"].isin(["extreme_payment_deviation", "moderate_payment_deviation", "duplicate_like_billing"]).astype(int)

    # Base masks
    clean_mask = (df["IS_ANOMALY_INJECTED"] == 0) & df["STAGE6_ENSEMBLE_RISK_SCORE"].notna()
    price_mask = df["IS_STAGE6_PRICE_SCENARIO"] == 1
    mod_mask = df["SCENARIO_TYPE"] == "moderate_payment_deviation"
    ext_mask = df["SCENARIO_TYPE"] == "extreme_payment_deviation"
    dup_mask = df["SCENARIO_TYPE"] == "duplicate_like_billing"

    flagged = df["STAGE6_AUDIT_FLAG"] == 1

    # --- Metrics ---
    clean_fpr = flagged[clean_mask].mean() if clean_mask.any() else 0

    # A. Price-scenario cohort
    price_cohort = df[clean_mask | price_mask]
    p_tp = (flagged & price_mask).sum()
    p_fp = (flagged & clean_mask).sum()
    p_fn = (~flagged & price_mask).sum()

    p_precision = p_tp / (p_tp + p_fp) if (p_tp + p_fp) > 0 else 0
    p_recall = p_tp / (p_tp + p_fn) if (p_tp + p_fn) > 0 else 0
    p_f1 = 2 * (p_precision * p_recall) / (p_precision + p_recall) if (p_precision + p_recall) > 0 else 0

    ext_recall = flagged[ext_mask].mean() if ext_mask.any() else 0
    mod_recall = flagged[mod_mask].mean() if mod_mask.any() else 0

    # B. All-scenario cohort
    overall_recall = flagged[df["IS_STAGE6_INJECTED_SCENARIO"] == 1].mean()
    dup_recall = flagged[dup_mask].mean() if dup_mask.any() else 0

    print("\n=== STAGE 6 EVALUATION METRICS ===")
    print(f"Clean False Positive Rate:      {clean_fpr:.2%}")
    print("\n-- A. Price Scenario Cohort --")
    print(f"Precision:                      {p_precision:.2%}")
    print(f"Recall:                         {p_recall:.2%}")
    print(f"F1 Score:                       {p_f1:.2%}")
    print(f"Extreme Scenario Recall:        {ext_recall:.2%}")
    print(f"Moderate Scenario Recall:       {mod_recall:.2%}")

    print("\n-- B. All-Scenario Cohort --")
    print(f"Overall Scenario Recall:        {overall_recall:.2%}")
    print(f"Duplicate-like Billing Recall:  {dup_recall:.2%} (Note: No relational model exists in this POC)")

    # --- C. Queue-Ranking Evaluation ---
    def calc_queue(mask_eval, q_pct, label_col):
        eval_df = df[mask_eval].sort_values("STAGE6_ENSEMBLE_RISK_SCORE", ascending=False)
        q_size = max(1, int(len(eval_df) * q_pct))
        queue = eval_df.head(q_size)

        true_anom = queue[label_col].sum()
        total_anom = eval_df[label_col].sum()
        clean_claims = len(queue) - true_anom

        precision = true_anom / len(queue) if len(queue) > 0 else 0
        recall = true_anom / total_anom if total_anom > 0 else 0

        return len(queue), true_anom, clean_claims, precision, recall

    # Setup queue arrays
    q_metrics = []

    # Price
    q_metrics.append(["Price scenarios", "Top 5%"] + list(calc_queue(clean_mask | price_mask, 0.05, "IS_STAGE6_PRICE_SCENARIO")))
    q_metrics.append(["Price scenarios", "Top 10%"] + list(calc_queue(clean_mask | price_mask, 0.10, "IS_STAGE6_PRICE_SCENARIO")))

    # All
    q_metrics.append(["All injected scenarios", "Top 5%"] + list(calc_queue(clean_mask | (df["IS_STAGE6_INJECTED_SCENARIO"] == 1), 0.05, "IS_STAGE6_INJECTED_SCENARIO")))
    q_metrics.append(["All injected scenarios", "Top 10%"] + list(calc_queue(clean_mask | (df["IS_STAGE6_INJECTED_SCENARIO"] == 1), 0.10, "IS_STAGE6_INJECTED_SCENARIO")))

    q_df = pd.DataFrame(q_metrics, columns=["Evaluation cohort", "Queue", "Claims in queue", "Scenarios captured", "Clean claims included", "Precision", "Recall"])
    q_df["Precision"] = q_df["Precision"].map("{:.2%}".format)
    q_df["Recall"] = q_df["Recall"].map("{:.2%}".format)

    print("\n-- C. Queue-Ranking Evaluation --")
    print(q_df.to_string(index=False))

    # --- 6. Compare Scores & Clusters ---
    print("\n-- Score Comparison by Scenario --")
    scen_agg = df.groupby("SCENARIO_TYPE", dropna=False).agg(
        Count=("STAGE6_ENSEMBLE_RISK_SCORE", "count"),
        Median_S4=("STAGE4_PRICE_RISK_SCORE", "median"),
        Median_S5=("STAGE5_IF_RISK_SCORE", "median"),
        Median_Ens=("STAGE6_ENSEMBLE_RISK_SCORE", "median"),
        Flag_Rate=("STAGE6_AUDIT_FLAG", "mean")
    ).reset_index()
    scen_agg["Flag_Rate"] = scen_agg["Flag_Rate"].map("{:.2%}".format)
    print(scen_agg.to_string(index=False))

    if "STAGE5_CLUSTER_KEY" in df.columns:
        print("\n-- Cluster-Level Diagnostics --")

        # Calculate cluster-level metrics safely
        c_agg = df.groupby("STAGE5_CLUSTER_KEY").apply(lambda g: pd.Series({
            "Clean claims": (g["IS_ANOMALY_INJECTED"] == 0).sum(),
            "Injected scenarios": (g["IS_ANOMALY_INJECTED"] == 1).sum(),
            "Median Ensemble Score": g["STAGE6_ENSEMBLE_RISK_SCORE"].median(),
            "Clean FPR": g.loc[g["IS_ANOMALY_INJECTED"] == 0, "STAGE6_AUDIT_FLAG"].mean() if (g["IS_ANOMALY_INJECTED"] == 0).any() else 0,
            "Price-scenario Recall": g.loc[g["IS_STAGE6_PRICE_SCENARIO"] == 1, "STAGE6_AUDIT_FLAG"].mean() if (g["IS_STAGE6_PRICE_SCENARIO"] == 1).any() else 0,
            "Flagged Count": g["STAGE6_AUDIT_FLAG"].sum()
        })).reset_index()

        c_agg["Clean FPR"] = c_agg["Clean FPR"].map("{:.2%}".format)
        c_agg["Price-scenario Recall"] = c_agg["Price-scenario Recall"].map("{:.2%}".format)
        print(c_agg.to_string(index=False))

def main():
    logger.info("Starting Stage 6: Transparent Parallel Risk-Fusion Combiner")

    df = load_and_validate(INPUT_FILE)
    df = define_valid_signals(df)
    df = fuse_scores(df)
    df = calibrate_final_tiers(df)

    run_quality_assertions(df)
    evaluate_performance(df)

    os.makedirs(OUT_DIR, exist_ok=True)

    # Save Output Fields
    out_cols = list(df.columns) # We retain all upstream columns and append stage 6
    logger.info(f"Saving {len(df)} records to Parquet & CSV...")
    df.to_parquet(OUT_PARQUET, index=False)
    df.to_csv(OUT_CSV, index=False)

    logger.info("Stage 6 completed successfully.")

if __name__ == "__main__":
    main()


=== STAGE 6 EVALUATION METRICS ===
Clean False Positive Rate:      2.67%

-- A. Price Scenario Cohort --
Precision:                      28.73%
Recall:                         42.28%
F1 Score:                       34.21%
Extreme Scenario Recall:        64.56%
Moderate Scenario Recall:       2.27%

-- B. All-Scenario Cohort --
Overall Scenario Recall:        30.48%
Duplicate-like Billing Recall:  2.86% (Note: No relational model exists in this POC)

-- C. Queue-Ranking Evaluation --
     Evaluation cohort   Queue  Claims in queue  Scenarios captured  Clean claims included Precision Recall
       Price scenarios  Top 5%              495                 123                    372    24.85% 50.00%
       Price scenarios Top 10%              990                 185                    805    18.69% 75.20%
All injected scenarios  Top 5%              500                 127                    373    25.40% 36.18%
All injected scenarios Top 10%             1001                 197            

LAYER 2

In [16]:
!pip install -q langchain-groq langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.1 MB/s eta 0:00:00


In [17]:
import os
import json
from typing import TypedDict, Dict, Any
from google.colab import userdata
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

In [18]:
# 1. Initialize Groq API Key from Colab Secrets or Environment
try:
    groq_key = userdata.get('GROQ_API_KEY')
    os.environ["GROQ_API_KEY"] = groq_key
    print("Groq API Key loaded successfully!")
except Exception:
    # Prompt user if secret is not set in Colab sidebar
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Initialize Llama 3.3 70B via Groq
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.1 # Low temperature for reliable audit reasoning
)

Groq API Key loaded successfully!


In [22]:
"""
Stage 7: ClaimSignal Evidence-Grounded Review Engine
Converts Stage 4, Stage 5, and Stage 6 model evidence into a concise,
grounded, human-reviewable payment-integrity memo using Llama-3.3-70b-versatile via Groq.
"""

import os
import json
import logging
import pandas as pd
from typing import List, Dict, Any, Tuple, Optional
from pydantic import BaseModel, Field, ValidationError

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.exceptions import OutputParserException

# Configure logging
logger = logging.getLogger(__name__)

# =========================================================================
# 1. CONSTANTS & APPROVED SIGNALS
# =========================================================================

APPROVED_SIGNALS = [
    "Audit Priority Score",
    "Payment Deviation",
    "Stage 4 Price Risk",
    "Stage 5 Multivariate Risk",
    "Data Quality Status"
]

PROHIBITED_TERMS = [
    "fraud", "fraudulent", "improper", "overcharged", "deny",
    "denial", "reject", "illegal", "guilty", "overpayment confirmed",
    "auto-clear", "payment action"
]

# =========================================================================
# 2. PYDANTIC SCHEMAS (STRUCTURED OUTPUT)
# =========================================================================

class EvidenceItem(BaseModel):
    signal: str = Field(
        description="Must be exactly one approved human-readable signal name."
    )
    finding: str = Field(
        description="Concise factual finding using only supplied evidence values."
    )
    source_fields: List[str] = Field(
        description="Exact CLAIM_EVIDENCE field names supporting the finding."
    )

class ClaimReviewMemo(BaseModel):
    review_recommendation: str = Field(
        description="Must exactly match required_review_recommendation from the evidence packet."
    )
    priority_summary: str = Field(
        description="One or two concise factual sentences."
    )
    evidence: List[EvidenceItem] = Field(
        description="Two to four evidence items in descending importance."
    )
    recommended_checks: List[str] = Field(
        description="Two to four human verification checks only."
    )
    data_quality_status: str = Field(
        description="State whether data context is complete or needs review."
    )
    limitations: str = Field(
        description="Mandatory POC and human-review limitation statement."
    )


# =========================================================================
# 3. SYSTEM PROMPT
# =========================================================================

SYSTEM_PROMPT = """You are ClaimSignal Review Assistant, assisting a healthcare
payment-integrity analyst.

Your only task is to convert supplied CLAIM_EVIDENCE into a concise,
auditable analyst review memo.

You do not make payment decisions and do not determine fraud, billing error,
medical necessity, coding correctness, contract compliance, overpayment, or
payment action.

Rules:
1. Use only facts explicitly present in CLAIM_EVIDENCE.
2. Do not calculate, estimate, infer, or invent missing values.
3. Set review_recommendation exactly equal to required_review_recommendation.
4. Use only these exact evidence signal names:
   - Audit Priority Score
   - Payment Deviation
   - Stage 4 Price Risk
   - Stage 5 Multivariate Risk
   - Data Quality Status
5. Use machine-readable evidence-packet field names only in source_fields.
6. If policy_text_available is false, do not name or claim violation of any
   policy, contract, fee schedule, or documentation requirement.
7. Recommend verification steps only; never payment actions.
8. If data_quality_review is true, prioritize data-quality review and do not
   recommend payment-integrity escalation.
9. Do not use prohibited or accusatory language.
10. Do not provide chain-of-thought, hidden reasoning, or internal
    step-by-step reasoning.
11. Return only the requested Pydantic structured output."""


# =========================================================================
# 4. EVIDENCE PACKET BUILDER & DETERMINISTIC ROUTING
# =========================================================================

def _safe_get(row: pd.Series, col: str, default: Any = None) -> Any:
    """Safely retrieve a value from a pandas Series without NaN issues."""
    if col not in row.index or pd.isna(row[col]):
        return default
    val = row[col]
    if hasattr(val, "item"):
        return val.item()
    return val

def get_required_recommendation(packet: dict) -> str:
    """Deterministic routing rules for the review recommendation."""
    if packet.get("data_quality_review") is True or packet.get("missing_hcpcs_flag") is True:
        return "Route to data-quality review before payment-integrity analysis."

    tier = str(packet.get("audit_priority_tier", "")).lower()
    if tier in ["elevated", "extreme"]:
        return "Prioritize for payment-integrity analyst review."

    return "Not prioritized for payment-integrity review; continue standard workflow."

def build_evidence_packet(row: pd.Series) -> Dict[str, Any]:
    """Constructs the structured evidence packet strictly using Stage 6 output fields."""
    claim_id = _safe_get(row, 'CLM_ID', _safe_get(row, 'CLAIM_LINE_ID'))
    proc_code = _safe_get(row, 'HCPCS_CD_1', _safe_get(row, 'ORIGINAL_HCPCS_CD'))
    stage4_score = _safe_get(row, 'STAGE4_PRICE_RISK_SCORE')
    stage5_score = _safe_get(row, 'STAGE5_IF_RISK_SCORE')

    dq_flag = bool(_safe_get(row, 'STAGE6_DATA_QUALITY_REVIEW', False))
    missing_hcpcs_flag = bool(_safe_get(row, 'STAGE5_MISSING_HCPCS_FLAG', False))

    is_dq_review = (
        dq_flag or
        missing_hcpcs_flag or
        proc_code is None or
        (stage4_score is None and stage5_score is None)
    )

    packet = {
        "claim_identifier": str(claim_id) if claim_id is not None else None,
        "procedure_code": str(proc_code) if proc_code is not None else None,
        "cluster_segment": _safe_get(row, 'STAGE5_CLUSTER_KEY'),

        "working_payment_amount": _safe_get(row, 'WORKING_PMT_AMT'),
        "expected_payment_amount": _safe_get(row, 'EXPECTED_PMT_FINAL'),
        "payment_residual_amount": _safe_get(row, 'PAYMENT_RESIDUAL_FINAL'),
        "relative_surge_ratio": _safe_get(row, 'RELATIVE_SURGE_RATIO_FINAL'),
        "payment_ratio_to_expected": _safe_get(row, 'PAYMENT_RATIO_TO_EXPECTED_FINAL'),

        "stage4_price_risk_score": stage4_score,
        "stage4_price_tier": _safe_get(row, 'STAGE4_PRICE_TIER'),

        "stage5_multivariate_risk_score": stage5_score,
        "stage5_anomaly_tier": _safe_get(row, 'STAGE5_IF_TIER'),

        "ensemble_risk_score": _safe_get(row, 'STAGE6_ENSEMBLE_RISK_SCORE'),
        "audit_priority_score": _safe_get(row, 'AUDIT_PRIORITY_SCORE'),
        "audit_priority_tier": _safe_get(row, 'STAGE6_AUDIT_TIER'),
        "fusion_source": _safe_get(row, 'STAGE6_FUSION_SOURCE'),

        "data_quality_review": is_dq_review,
        "missing_hcpcs_flag": missing_hcpcs_flag,

        "policy_text_available": False,
        "policy_text": None,

        "required_review_recommendation": None,

        "limitations": [
            "Synthetic/de-identified CMS claims POC data",
            "Scores prioritize claims for human review and do not establish billing error, fraud, medical necessity, overpayment, or payment action"
        ]
    }

    packet["required_review_recommendation"] = get_required_recommendation(packet)
    return packet


# =========================================================================
# 5. SAFETY & GROUNDING GUARDRAIL VALIDATOR
# =========================================================================

def validate_memo_output(evidence_packet: Dict[str, Any], memo_obj: ClaimReviewMemo) -> Tuple[bool, List[str], Dict[str, Any]]:
    """
    Validates LLM output against strict grounding, safety, and routing guardrails.
    Returns (is_valid, validation_errors, memo_dict).
    """
    errors = []
    memo_dict = memo_obj.model_dump()

    # 1. Prohibited Terms Check (excluding the standard limitations string)
    check_dict = {k: v for k, v in memo_dict.items() if k != 'limitations'}
    memo_text_block = json.dumps(check_dict).lower()

    found_prohibited = [term for term in PROHIBITED_TERMS if term in memo_text_block]
    if found_prohibited:
        errors.append(f"Safety Violation: Prohibited terms detected: {found_prohibited}")

    # 2. Recommendation Match Check
    required_rec = evidence_packet.get("required_review_recommendation")
    if memo_obj.review_recommendation != required_rec:
        errors.append(
            f"Routing Error: review_recommendation '{memo_obj.review_recommendation}' "
            f"does not exactly match required_review_recommendation '{required_rec}'."
        )

    # 3. Evidence Items Count and Signal Validation
    if len(memo_obj.evidence) > 4:
        errors.append(f"Schema Violation: Maximum 4 evidence items allowed, found {len(memo_obj.evidence)}.")

    packet_keys = set(evidence_packet.keys())
    for ev in memo_obj.evidence:
        # Validate approved signal name
        if ev.signal not in APPROVED_SIGNALS:
            errors.append(f"Grounding Error: Signal '{ev.signal}' is not in APPROVED_SIGNALS.")

        # Validate source fields existence and null values
        for sf in ev.source_fields:
            if sf not in packet_keys:
                errors.append(f"Grounding Error: Source field '{sf}' does not exist in the evidence packet.")
            elif evidence_packet.get(sf) is None:
                errors.append(f"Grounding Error: Source field '{sf}' is cited but is None in the evidence packet.")

    # 4. Data Quality Routing Constraints
    if evidence_packet.get("data_quality_review") is True:
        if "data-quality review" not in memo_obj.review_recommendation.lower():
            errors.append("Routing Error: data_quality_review is True but recommendation missing 'data-quality review'.")
        if "analyst review" in memo_obj.review_recommendation.lower() and "data-quality" not in memo_obj.review_recommendation.lower():
            errors.append("Routing Error: Cannot recommend payment-integrity analyst review for data-quality claims.")

    # 5. Policy Hallucination Check
    if evidence_packet.get("policy_text_available") is False:
        disallowed_policy_terms = ["violated policy", "policy violation", "contract violation", "fee schedule violation"]
        for term in disallowed_policy_terms:
            if term in memo_text_block:
                errors.append(f"Grounding Error: Referenced '{term}' when policy_text_available is False.")

    is_valid = len(errors) == 0
    return is_valid, errors, memo_dict


# =========================================================================
# 6. CORE REUSABLE ENGINE FUNCTION
# =========================================================================

def generate_claim_review_memo(selected_claim_row: pd.Series) -> Dict[str, Any]:
    """
    Main entry point for generating a single claim review memo.
    Constructs packet, calls Groq LLM, validates output, and returns structured result.
    """
    model_name = "llama-3.3-70b-versatile"

    # Check for API key securely without exposing environment secrets
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        return {
            "evidence_packet": {},
            "memo": None,
            "validation_passed": False,
            "validation_errors": ["System Error: GROQ_API_KEY environment variable is not set."],
            "model_name": model_name
        }

    # Build Evidence Packet
    packet = build_evidence_packet(selected_claim_row)

    # Initialize LLM with Graceful Exception Handling
    try:
        llm = ChatGroq(
            model_name=model_name,
            temperature=0.1,
            max_retries=2
        )
        structured_llm = llm.with_structured_output(ClaimReviewMemo)

        prompt_content = f"CLAIM_EVIDENCE:\n{json.dumps(packet, indent=2)}"
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=prompt_content)
        ]

        # Single claim invocation
        response_memo = structured_llm.invoke(messages)

    except OutputParserException as e:
        return {
            "evidence_packet": packet,
            "memo": None,
            "validation_passed": False,
            "validation_errors": [f"Parsing Error: Model output failed structured Pydantic parsing."],
            "model_name": model_name
        }
    except Exception as e:
        error_msg = str(e)
        if "rate limit" in error_msg.lower():
            safe_error = "API Error: Groq rate limit exceeded. Please try again later."
        else:
            safe_error = "API Error: Internal API call failure."
            logger.error(f"Groq API Error: {error_msg}")

        return {
            "evidence_packet": packet,
            "memo": None,
            "validation_passed": False,
            "validation_errors": [safe_error],
            "model_name": model_name
        }

    # Validate output grounding & guardrails
    is_valid, validation_errors, clean_memo = validate_memo_output(packet, response_memo)

    return {
        "evidence_packet": packet,
        "memo": clean_memo if response_memo else None,
        "validation_passed": is_valid,
        "validation_errors": validation_errors,
        "model_name": model_name
    }

In [23]:
import os
import json
import pandas as pd
# from src.llm_copilot import generate_claim_review_memo

def run_stage7_demo():
    print("=" * 80)
    print("=== STAGE 7: REVIEW ENGINE DEMO (SINGLE SELECTED CLAIM) ===")
    print("=" * 80)

    data_path = "data/processed/cms_claims_stage6_ensemble_scored.parquet"

    if not os.path.exists(data_path):
        print(f"Error: Dataset not found at '{data_path}'. Check your workspace path.")
        return

    # Load Stage 6 scored dataset
    df = pd.read_parquet(data_path)

    # Filter for real elevated or extreme claims with complete data context
    demo_candidates = df[
        (df["STAGE6_AUDIT_TIER"].isin(["elevated", "extreme"])) &
        (df["STAGE6_DATA_QUALITY_REVIEW"] == False)
    ].sort_values("AUDIT_PRIORITY_SCORE", ascending=False)

    if demo_candidates.empty:
        print("Warning: No elevated/extreme claims found without data quality flags. Falling back to top score.")
        demo_claim = df.sort_values("AUDIT_PRIORITY_SCORE", ascending=False).iloc[0]
    else:
        demo_claim = demo_candidates.iloc[0]

    claim_id = demo_claim.get("CLM_ID", demo_claim.get("CLAIM_LINE_ID", "UNKNOWN"))
    print(f"\nSelected High-Priority Claim ID: {claim_id}\n")

    # Generate memo using core reusable function
    result = generate_claim_review_memo(demo_claim)

    print("1. STRUCTURED EVIDENCE PACKET:")
    print(json.dumps(result["evidence_packet"], indent=2))

    print("\n2. VALIDATION & GROUNDING STATUS:")
    if result["validation_passed"]:
        print(" [PASS] Output complies with schema, grounding rules, and safety filters.")
    else:
        print(" [FAIL] Grounding/Validation Errors:")
        for err in result["validation_errors"]:
            print(f"  - {err}")

    print("\n3. FINAL ANALYST-FACING REVIEW MEMO:")
    if result["validation_passed"] and result["memo"]:
        print(json.dumps(result["memo"], indent=2))
    else:
        print("Memo generation requires analyst review because output grounding validation did not pass.")

if __name__ == "__main__":
    run_stage7_demo()

=== STAGE 7: REVIEW ENGINE DEMO (SINGLE SELECTED CLAIM) ===

Selected High-Priority Claim ID: 887173385436285

1. STRUCTURED EVIDENCE PACKET:
{
  "claim_identifier": "887173385436285",
  "procedure_code": "92012",
  "cluster_segment": "1",
  "working_payment_amount": 550.0,
  "expected_payment_amount": 90.0,
  "payment_residual_amount": 460.0,
  "relative_surge_ratio": 5.111111111111111,
  "payment_ratio_to_expected": 6.111111111111111,
  "stage4_price_risk_score": 1.0,
  "stage4_price_tier": "extreme",
  "stage5_multivariate_risk_score": 1.0,
  "stage5_anomaly_tier": "extreme",
  "ensemble_risk_score": 1.0,
  "audit_priority_score": 100.0,
  "audit_priority_tier": "extreme",
  "fusion_source": "stage4_and_stage5",
  "data_quality_review": false,
  "missing_hcpcs_flag": false,
  "policy_text_available": false,
  "policy_text": null,
  "required_review_recommendation": "Prioritize for payment-integrity analyst review.",
  "limitations": [
    "Synthetic/de-identified CMS claims POC data

#Stage 8: Streamlit Dashboard

In [35]:
import os
os.makedirs("src", exist_ok=True)

In [36]:
%%writefile src/llm_copilot.py
import os
import json
import logging
import pandas as pd
from typing import List, Dict, Any, Tuple
from pydantic import BaseModel, Field

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.exceptions import OutputParserException

logger = logging.getLogger(__name__)

APPROVED_SIGNALS = [
    "Audit Priority Score",
    "Payment Deviation",
    "Stage 4 Price Risk",
    "Stage 5 Multivariate Risk",
    "Data Quality Status"
]

PROHIBITED_TERMS = [
    "fraud", "fraudulent", "improper", "overcharged", "deny",
    "denial", "reject", "illegal", "guilty", "overpayment confirmed",
    "auto-clear", "payment action"
]

class EvidenceItem(BaseModel):
    signal: str = Field(description="Must be exactly one approved human-readable signal name.")
    finding: str = Field(description="Concise factual finding using only supplied evidence values.")
    source_fields: List[str] = Field(description="Exact CLAIM_EVIDENCE field names supporting the finding.")

class ClaimReviewMemo(BaseModel):
    review_recommendation: str = Field(description="Must exactly match required_review_recommendation from the evidence packet.")
    priority_summary: str = Field(description="One or two concise factual sentences.")
    evidence: List[EvidenceItem] = Field(description="Two to four evidence items in descending importance.")
    recommended_checks: List[str] = Field(description="Two to four human verification checks only.")
    data_quality_status: str = Field(description="State whether data context is complete or needs review.")
    limitations: str = Field(description="Mandatory POC and human-review limitation statement.")

SYSTEM_PROMPT = """You are ClaimSignal Review Assistant, assisting a healthcare payment-integrity analyst.
Your only task is to convert supplied CLAIM_EVIDENCE into a concise, auditable analyst review memo.
You do not make payment decisions and do not determine fraud, billing error, medical necessity, coding correctness, contract compliance, overpayment, or payment action.

Rules:
1. Use only facts explicitly present in CLAIM_EVIDENCE.
2. Do not calculate, estimate, infer, or invent missing values.
3. Set review_recommendation exactly equal to required_review_recommendation.
4. Use only these exact evidence signal names:
   - Audit Priority Score
   - Payment Deviation
   - Stage 4 Price Risk
   - Stage 5 Multivariate Risk
   - Data Quality Status
5. Use machine-readable evidence-packet field names only in source_fields.
6. If policy_text_available is false, do not name or claim violation of any policy, contract, fee schedule, or documentation requirement.
7. Recommend verification steps only; never payment actions.
8. If data_quality_review is true, prioritize data-quality review and do not recommend payment-integrity escalation.
9. Do not use prohibited or accusatory language.
10. Return only the requested Pydantic structured output."""

def _safe_get(row: pd.Series, col: str, default: Any = None) -> Any:
    if col not in row.index or pd.isna(row[col]):
        return default
    val = row[col]
    if hasattr(val, "item"):
        return val.item()
    return val

def get_required_recommendation(packet: dict) -> str:
    if packet.get("data_quality_review") is True or packet.get("missing_hcpcs_flag") is True:
        return "Route to data-quality review before payment-integrity analysis."

    tier = str(packet.get("audit_priority_tier", "")).lower()
    if tier in ["elevated", "extreme"]:
        return "Prioritize for payment-integrity analyst review."

    return "Not prioritized for payment-integrity review; continue standard workflow."

def build_evidence_packet(row: pd.Series) -> Dict[str, Any]:
    claim_id = _safe_get(row, 'CLM_ID', _safe_get(row, 'CLAIM_LINE_ID'))
    proc_code = _safe_get(row, 'HCPCS_CD_1', _safe_get(row, 'ORIGINAL_HCPCS_CD'))
    stage4_score = _safe_get(row, 'STAGE4_PRICE_RISK_SCORE')
    stage5_score = _safe_get(row, 'STAGE5_IF_RISK_SCORE')

    dq_flag = bool(_safe_get(row, 'STAGE6_DATA_QUALITY_REVIEW', False))
    missing_hcpcs_flag = bool(_safe_get(row, 'STAGE5_MISSING_HCPCS_FLAG', False))

    is_dq_review = (
        dq_flag or
        missing_hcpcs_flag or
        proc_code is None or
        (stage4_score is None and stage5_score is None)
    )

    packet = {
        "claim_identifier": str(claim_id) if claim_id is not None else None,
        "procedure_code": str(proc_code) if proc_code is not None else None,
        "cluster_segment": _safe_get(row, 'STAGE5_CLUSTER_KEY'),

        "working_payment_amount": _safe_get(row, 'WORKING_PMT_AMT'),
        "expected_payment_amount": _safe_get(row, 'EXPECTED_PMT_FINAL'),
        "payment_residual_amount": _safe_get(row, 'PAYMENT_RESIDUAL_FINAL'),
        "relative_surge_ratio": _safe_get(row, 'RELATIVE_SURGE_RATIO_FINAL'),
        "payment_ratio_to_expected": _safe_get(row, 'PAYMENT_RATIO_TO_EXPECTED_FINAL'),

        "stage4_price_risk_score": stage4_score,
        "stage4_price_tier": _safe_get(row, 'STAGE4_PRICE_TIER'),

        "stage5_multivariate_risk_score": stage5_score,
        "stage5_anomaly_tier": _safe_get(row, 'STAGE5_IF_TIER'),

        "ensemble_risk_score": _safe_get(row, 'STAGE6_ENSEMBLE_RISK_SCORE'),
        "audit_priority_score": _safe_get(row, 'AUDIT_PRIORITY_SCORE'),
        "audit_priority_tier": _safe_get(row, 'STAGE6_AUDIT_TIER'),
        "fusion_source": _safe_get(row, 'STAGE6_FUSION_SOURCE'),

        "data_quality_review": is_dq_review,
        "missing_hcpcs_flag": missing_hcpcs_flag,

        "policy_text_available": False,
        "policy_text": None,
        "required_review_recommendation": None,

        "limitations": [
            "Synthetic/de-identified CMS claims POC data",
            "Scores prioritize claims for human review and do not establish billing error, fraud, medical necessity, overpayment, or payment action"
        ]
    }

    packet["required_review_recommendation"] = get_required_recommendation(packet)
    return packet

def validate_memo_output(evidence_packet: Dict[str, Any], memo_obj: ClaimReviewMemo) -> Tuple[bool, List[str], Dict[str, Any]]:
    errors = []
    memo_dict = memo_obj.model_dump()

    check_dict = {k: v for k, v in memo_dict.items() if k != 'limitations'}
    memo_text_block = json.dumps(check_dict).lower()

    found_prohibited = [term for term in PROHIBITED_TERMS if term in memo_text_block]
    if found_prohibited:
        errors.append(f"Safety Violation: Prohibited terms detected: {found_prohibited}")

    required_rec = evidence_packet.get("required_review_recommendation")
    if memo_obj.review_recommendation != required_rec:
        errors.append(
            f"Routing Error: review_recommendation '{memo_obj.review_recommendation}' "
            f"does not exactly match required_review_recommendation '{required_rec}'."
        )

    if len(memo_obj.evidence) > 4:
        errors.append(f"Schema Violation: Maximum 4 evidence items allowed, found {len(memo_obj.evidence)}.")

    packet_keys = set(evidence_packet.keys())
    for ev in memo_obj.evidence:
        if ev.signal not in APPROVED_SIGNALS:
            errors.append(f"Grounding Error: Signal '{ev.signal}' is not in APPROVED_SIGNALS.")
        for sf in ev.source_fields:
            if sf not in packet_keys:
                errors.append(f"Grounding Error: Source field '{sf}' does not exist in the evidence packet.")
            elif evidence_packet.get(sf) is None:
                errors.append(f"Grounding Error: Source field '{sf}' is cited but is None in the evidence packet.")

    if evidence_packet.get("data_quality_review") is True:
        if "data-quality review" not in memo_obj.review_recommendation.lower():
            errors.append("Routing Error: data_quality_review is True but recommendation missing 'data-quality review'.")

    is_valid = len(errors) == 0
    return is_valid, errors, memo_dict

def generate_claim_review_memo(selected_claim_row: pd.Series) -> Dict[str, Any]:
    model_name = "llama-3.3-70b-versatile"

    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        return {
            "evidence_packet": {},
            "memo": None,
            "validation_passed": False,
            "validation_errors": ["System Error: GROQ_API_KEY environment variable is not set."],
            "model_name": model_name
        }

    packet = build_evidence_packet(selected_claim_row)

    try:
        llm = ChatGroq(
            model_name=model_name,
            temperature=0.1,
            max_retries=2
        )
        structured_llm = llm.with_structured_output(ClaimReviewMemo)

        prompt_content = f"CLAIM_EVIDENCE:\n{json.dumps(packet, indent=2)}"
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=prompt_content)
        ]

        response_memo = structured_llm.invoke(messages)

    except OutputParserException:
        return {
            "evidence_packet": packet,
            "memo": None,
            "validation_passed": False,
            "validation_errors": ["Parsing Error: Model output failed structured Pydantic parsing."],
            "model_name": model_name
        }
    except Exception as e:
        return {
            "evidence_packet": packet,
            "memo": None,
            "validation_passed": False,
            "validation_errors": [f"API Error: {str(e)}"],
            "model_name": model_name
        }

    is_valid, validation_errors, clean_memo = validate_memo_output(packet, response_memo)

    return {
        "evidence_packet": packet,
        "memo": clean_memo if response_memo else None,
        "validation_passed": is_valid,
        "validation_errors": validation_errors,
        "model_name": model_name
    }

Writing src/llm_copilot.py


In [27]:
!pip -q install streamlit pyngrok pydantic pyarrow

In [26]:
!npm install localtunnel -g -q

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 2s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋

In [44]:
%%writefile app.py
import os
import pandas as pd
import streamlit as st
# from src.llm_copilot import generate_claim_review_memo
from src.llm_copilot import generate_claim_review_memo

# =========================================================================
# 0. PAGE CONFIGURATION
# =========================================================================
st.set_page_config(
    page_title="ClaimSignal | Payment-Integrity Review Queue",
    page_icon="🔎",
    layout="wide",
    initial_sidebar_state="collapsed"
)

# =========================================================================
# 1. DATA LOADING & HELPERS
# =========================================================================
@st.cache_data
def load_stage6_data():
    return pd.read_parquet("data/processed/cms_claims_stage6_ensemble_scored.parquet")

def get_identifier(row: pd.Series) -> str:
    for col in ['CLM_ID', 'CLAIM_LINE_ID', 'DESYNPUF_ID']:
        if col in row.index and not pd.isna(row[col]):
            return str(row[col])
    return "UNKNOWN"

def get_hcpcs(row: pd.Series) -> str:
    for col in ['HCPCS_CD_1', 'ORIGINAL_HCPCS_CD']:
        if col in row.index and not pd.isna(row[col]):
            return str(row[col])
    return "UNKNOWN"

def format_currency(val) -> str:
    if pd.isna(val): return "$0.00"
    return f"${float(val):,.2f}"

def format_ratio(val) -> str:
    if pd.isna(val): return "N/A"
    return f"{float(val):.2f}x"

df = load_stage6_data()

# =========================================================================
# 2. HEADER & GUARDRAIL STATEMENT
# =========================================================================
st.title("ClaimSignal | Payment-Integrity Review Queue")
st.markdown("##### Human-in-the-loop prioritization prototype for synthetic CMS claims")

st.info(
    "This prototype prioritizes claims for human review using payment deviation, "
    "price-risk, and multivariate anomaly signals. It does not determine fraud, "
    "billing error, medical necessity, overpayment, or payment action.",
    icon="⚠️"
)

st.markdown("---")

# =========================================================================
# 3. REVIEW-QUEUE SUMMARY METRICS
# =========================================================================
flagged_for_review = df[df["STAGE6_AUDIT_TIER"].isin(["elevated", "extreme"])]
extreme_claims = df[df["STAGE6_AUDIT_TIER"] == "extreme"]
elevated_claims = df[df["STAGE6_AUDIT_TIER"] == "elevated"]

col1, col2, col3, col4 = st.columns(4)
with col1:
    st.metric("Total Claims", f"{len(df):,}")
with col2:
    st.metric(
        "Claims Flagged for Review",
        f"{len(flagged_for_review):,}",
        help="Elevated or extreme audit-priority tier; analyst review queue only."
    )
    st.caption("Elevated or extreme audit-priority tier; analyst review queue only.")
with col3:
    st.metric("Extreme Priority Claims", f"{len(extreme_claims):,}")
with col4:
    st.metric("Elevated Priority Claims", f"{len(elevated_claims):,}")

st.markdown("---")

# =========================================================================
# 4. FLAGGED CLAIMS REVIEW QUEUE
# =========================================================================
st.subheader("Flagged Claims Review Queue")

# Sort descending by priority score
queue_df = flagged_for_review.sort_values("AUDIT_PRIORITY_SCORE", ascending=False).copy()

# Build display dataframe
display_data = []
for _, row in queue_df.iterrows():
    display_data.append({
        "Claim ID": get_identifier(row),
        "Procedure Code": get_hcpcs(row),
        "Audit Priority Score": round(float(row.get("AUDIT_PRIORITY_SCORE", 0.0)), 2),
        "Audit Tier": str(row.get("STAGE6_AUDIT_TIER", "")).capitalize(),
        "Working Payment": format_currency(row.get("WORKING_PMT_AMT")),
        "Expected Payment": format_currency(row.get("EXPECTED_PMT_FINAL")),
        "Payment Ratio": format_ratio(row.get("PAYMENT_RATIO_TO_EXPECTED_FINAL")),
        "Data Quality Review": "Yes" if row.get("STAGE6_DATA_QUALITY_REVIEW") else "No"
    })

display_df = pd.DataFrame(display_data)

st.dataframe(
    display_df,
    use_container_width=True,
    hide_index=True
)

st.markdown("---")

# =========================================================================
# 5. CLAIM SELECTION
# =========================================================================
st.subheader("Claim Selection")

def build_label(row: pd.Series) -> str:
    cid = get_identifier(row)
    hcpcs = get_hcpcs(row)
    score = round(float(row.get("AUDIT_PRIORITY_SCORE", 0.0)), 1)
    tier = str(row.get("STAGE6_AUDIT_TIER", "")).capitalize()
    return f"Claim {cid} | HCPCS {hcpcs} | Priority {score} | {tier}"

claim_options = {build_label(row): i for i, row in queue_df.iterrows()}
selected_label = st.selectbox("Select a flagged claim to review:", list(claim_options.keys()))

selected_idx = claim_options[selected_label]
selected_claim_row = queue_df.loc[selected_idx]
current_claim_id = get_identifier(selected_claim_row)

# Reset state if a new claim is selected
if "last_claim_id" not in st.session_state or st.session_state.last_claim_id != current_claim_id:
    st.session_state.last_claim_id = current_claim_id
    st.session_state.memo_result = None

# =========================================================================
# 6. SELECTED CLAIM EVIDENCE PANEL
# =========================================================================
col_left, col_right = st.columns(2)

with col_left:
    st.markdown("**Selected Claim Context**")
    context_data = {
        "Claim Identifier": current_claim_id,
        "Procedure Code": get_hcpcs(selected_claim_row),
        "Cluster Segment": str(selected_claim_row.get("STAGE5_CLUSTER_KEY", "N/A")),
        "Working Payment": format_currency(selected_claim_row.get("WORKING_PMT_AMT")),
        "Expected Payment": format_currency(selected_claim_row.get("EXPECTED_PMT_FINAL")),
        "Payment Residual": format_currency(selected_claim_row.get("PAYMENT_RESIDUAL_FINAL")),
        "Payment Ratio to Expected": format_ratio(selected_claim_row.get("PAYMENT_RATIO_TO_EXPECTED_FINAL")),
        "Audit Priority Score": round(float(selected_claim_row.get("AUDIT_PRIORITY_SCORE", 0.0)), 2),
        "Audit Priority Tier": str(selected_claim_row.get("STAGE6_AUDIT_TIER", "")).capitalize()
    }
    st.dataframe(pd.DataFrame(list(context_data.items()), columns=["Attribute", "Value"]), hide_index=True, use_container_width=True)

with col_right:
    st.markdown("**Deterministic Review Signals**")

    pmt_resid = format_currency(selected_claim_row.get("PAYMENT_RESIDUAL_FINAL"))
    pmt_ratio = format_ratio(selected_claim_row.get("PAYMENT_RATIO_TO_EXPECTED_FINAL"))
    stage4_score = selected_claim_row.get("STAGE4_PRICE_RISK_SCORE")
    stage5_score = selected_claim_row.get("STAGE5_IF_RISK_SCORE")
    dq_flag = selected_claim_row.get("STAGE6_DATA_QUALITY_REVIEW")

    signals_data = [
        {"Signal": "Payment Deviation", "Value": f"{pmt_resid} | {pmt_ratio}", "Tier / Status": "Not applicable"},
        {"Signal": "Stage 4 Price Risk", "Value": f"{stage4_score:.2f}" if pd.notna(stage4_score) else "None", "Tier / Status": str(selected_claim_row.get("STAGE4_PRICE_TIER", "N/A")).capitalize()},
        {"Signal": "Stage 5 Multivariate Risk", "Value": f"{stage5_score:.2f}" if pd.notna(stage5_score) else "None", "Tier / Status": str(selected_claim_row.get("STAGE5_IF_TIER", "N/A")).capitalize()},
        {"Signal": "Audit Priority Score", "Value": f"{selected_claim_row.get('AUDIT_PRIORITY_SCORE', 0.0):.2f}", "Tier / Status": str(selected_claim_row.get("STAGE6_AUDIT_TIER", "")).capitalize()},
        {"Signal": "Data Quality Status", "Value": "Routed to review" if dq_flag else "Complete context", "Tier / Status": "Review" if dq_flag else "No routing flag"}
    ]
    st.dataframe(pd.DataFrame(signals_data), hide_index=True, use_container_width=True)

st.markdown("---")

# =========================================================================
# 7. STAGE 7 GROUNDED MEMO ACTION
# =========================================================================
if not os.environ.get("GROQ_API_KEY"):
    st.error("Groq API key is not configured. Set GROQ_API_KEY before generating a memo.")
else:
    if st.button("Generate Grounded Review Memo", type="primary"):
        with st.spinner("Generating and validating grounded review memo..."):
            try:
                result = generate_claim_review_memo(selected_claim_row)
                st.session_state.memo_result = result
            except Exception as e:
                st.error("An error occurred during memo generation. Please try again.")
                with st.expander("Technical details"):
                    st.write(str(e))

    st.caption("Generates one evidence-grounded summary for the selected claim.")

# =========================================================================
# 8. VALIDATION & MEMO PANEL
# =========================================================================
if st.session_state.get("memo_result"):
    result = st.session_state.memo_result

    st.subheader("Grounding & Safety Validation")

    if result.get("validation_passed") is True:
        st.success("✅ Grounding validation passed")

        st.markdown("### Analyst-Facing Review Memo")
        memo = result.get("memo", {})

        # Display Memo Sections
        st.markdown(f"**Review Recommendation:**\n> {memo.get('review_recommendation', '')}")
        st.markdown(f"**Priority Summary:**\n{memo.get('priority_summary', '')}")

        st.markdown("**Evidence:**")
        evidence_list = memo.get('evidence', [])
        if evidence_list:
            ev_df = pd.DataFrame([{
                "Approved Signal": e.get("signal", ""),
                "Finding": e.get("finding", ""),
                "Source Fields": ", ".join(e.get("source_fields", []))
            } for e in evidence_list])
            st.table(ev_df)

        st.markdown("**Recommended Verification Checks:**")
        for check in memo.get('recommended_checks', []):
            st.markdown(f"- {check}")

        st.markdown(f"**Data Quality Status:** {memo.get('data_quality_status', '')}")

        with st.expander("Limitations"):
            st.write(memo.get('limitations', ''))

    else:
        st.error("⚠ Memo withheld: grounding or safety validation failed.")
        with st.expander("Validation details"):
            for err in result.get("validation_errors", []):
                st.write(f"- {err}")

# =========================================================================
# 9. PERSISTENT LIMITATIONS FOOTER
# =========================================================================
st.markdown("---")
st.caption(
    "Synthetic/de-identified CMS claims POC data. Scores prioritize claims for "
    "human review and do not establish billing error, fraud, medical necessity, "
    "overpayment, or payment action."
)

Overwriting app.py


In [45]:
!streamlit run app.py --server.port 8501 &>/content/streamlit.log &
!sleep 3
!cat /content/streamlit.log



2026-08-14 03:26:53.744 Port 8501 is not available


In [46]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8501 > /content/cloudflared.log 2>&1 &
!sleep 5
!echo "Dashboard is ready. Access your temporary tunnel URL below:"
!grep -o 'https://[^[:space:]]*\.trycloudflare.com' /content/cloudflared.log

# Note: This URL is temporary and publicly accessible while the Colab runtime and
# tunnel process remain active. Do not share it widely, because visitors could
# trigger Groq requests. Stop the tunnel after recording the presentation.

cloudflared: Text file busy
Dashboard is ready. Access your temporary tunnel URL below:
https://catalogs-nursery-exam-two.trycloudflare.com
